# Notebook 3: ML Benchmark Experiment — GPCR Benchmark Panel

**Purpose:** The entire machine-learning benchmarking experiment for this
project, run identically across all **5 GPCR targets** (DRD2, CB2, ADORA2A,
OPRM1, CCR5) **x 3 activity pools** (`ki` / `ki_ic50` / `full`) = **15
target-pool combinations**, with no per-target or per-pool asymmetry
anywhere (all combinations use the same trial count, repeat count, and feature representations).

This single notebook absorbs what the original 8-notebook sketch called
`04_consensus_uncertainty_ad` and `08_validation_analysis` — calibration,
conformal prediction, and applicability domain are properties of the
trained model, not separate concerns, so they live here alongside the
models that produce them, rather than being split into separate
uncertainty/validation notebooks the way earlier implementations did.

**Required inputs** (all from notebook 02, already run and frozen):
- `data/processed/cleaned_data_{target}_{pool}.csv` for all 5 targets x 3 pools
  (lowercase target keys: `drd2`, `cb2`, `adora2a`, `oprm1`, `ccr5`)
- `data/processed/compound_scaffold_assignments.csv` (per-compound
  `global_scaffold_id`, `within_dataset_scaffold_rank_id`, `scaffold_size`,
  `is_singleton_scaffold` — scaffold groups are loaded from here, never
  recomputed independently, so notebook 03's train/test split can never
  silently drift from notebook 02's diversity reporting)
- `data/processed/dataset_summary_all_targets_pools.csv`,
  `data/processed/scaffold_diversity_summary.csv`
- `data/processed/manifest_02_data_cleaning.json` (re-hashed and verified in
  Module A, not just trusted by filename)

**Feature representations (exactly 3, bounded by design — not an exhaustive
fingerprint study):** Morgan fingerprints (radius 2, 2048 bits), physicochemical
descriptors (the 10 already computed in notebook 02: MW, LogP, TPSA, HBD,
HBA, RotBonds, HeavyAtomCount, RingCount, AromaticRingCount, FractionCSP3 —
loaded from the cleaned CSVs, never recomputed), and the two combined.

**Algorithms (exactly 3, classification AND regression variants of each):**
Random Forest, XGBoost, LightGBM.

**Hyperparameter policy:** equal Optuna tuning budget across all 5 targets —
**50 trials per (target, pool, model, task)**, tuned once per (target, pool)
on the combined feature representation and reused across all 3 feature
representations for the ablation comparison. This is a deliberate design
choice, not a shortcut: holding hyperparameters fixed across the
feature-representation comparison isolates the effect of the input features
from the effect of re-discovered hyperparameters, which is what the
ablation is actually meant to measure (see Module D for the full
rationale). By default no fixed-hyperparameter path is used; a `TUNING_MODE
= 'fixed'` toggle (config cell) can switch the whole notebook to documented
fixed hyperparameters for a no-tuning run — still leak-free, disclosed as such — but 'optuna' is the default and primary policy for this notebook — every reported model came from this tuning budget.

**Out of scope for this notebook (deliberately):** cross-target transfer
(train on target A, predict target B) — belongs to a later, optional
notebook 3B, built only after this notebook's artifacts (feature
definitions, global compound/scaffold IDs, frozen models, scalers) are
confirmed working, so 3B never needs to retrain anything from here.
DrugBank screening — kept in a separate notebook 4, since screening and
training both take hours and are usually rerun independently of one
another.

**Generated outputs** (all in `ml/features/`, `ml/models/`, `ml/results/`,
manifest + SHA-256 hashes in `ml/results/manifest_03_ml_benchmark.json`):
- Persisted feature arrays per (target, pool, representation)
- One tidy long-format benchmark results table spanning every
  (target x pool x representation x algorithm x task x repeat x metric)
  combination — `ml/results/benchmark_results_long.csv`
- Calibration, conformal-prediction, and applicability-domain summaries and
  frozen artifacts (one AD reference per target/pool/representation, fit
  once, never recomputed downstream)
- SHAP values, fingerprint-bit SMARTS decoding, feature-importance and
  hyperparameter-stability analyses
- Wilcoxon signed-rank tests + bootstrap CIs + effect sizes, per (target, pool)
- Final trained models, scalers, calibration/conformal objects, AD
  references, hyperparameters, and train/test compound IDs for every
  (target, pool, representation) — everything a later transfer-analysis
  notebook (3B) or the DrugBank-screening notebook (4) would need, without
  retraining
- Learning curves, persistent-misclassification error analysis,
  scaffold-level performance breakdown, prediction-difficulty analysis

**Design safeguards**
(full list in the notebook's closing summary cell): Y-randomization now takes the
actual best-performing model as a parameter instead of hardcoding XGBoost;
class_weight/scale_pos_weight are wired into the ONLY hyperparameter path
that exists now (no default path that silently drops them); applicability
domain is computed once per (target, pool, representation) and frozen,
never recomputed at a later stage; package versions are enumerated from the
actual import list programmatically; every bootstrap/SHAP/plot-source array
is persisted, not just the rendered figure; no on-figure titles or
code-identifier labels; full model names throughout (no `RF`/`XGB`/`LGB`
abbreviations); no numbered figure/table references in any printed or saved
text.


In [ ]:
import os
import multiprocessing

HPC_MODE = True

if HPC_MODE:
    N_CORES = int(os.environ.get("NCPUS") or os.environ.get("PBS_NP") or
                  os.environ.get("PBS_NCPUS") or os.environ.get("SLURM_CPUS_PER_TASK") or
                  multiprocessing.cpu_count())
    import matplotlib
    matplotlib.use("Agg")
else:
    N_CORES = min(multiprocessing.cpu_count(), 4)

# Keep math libraries single-threaded; joblib/sklearn CV uses N_CORES workers.
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1" if HPC_MODE else str(N_CORES)

os.environ["OMP_NESTED"] = "FALSE"
os.environ["MKL_DYNAMIC"] = "FALSE"

ENV = "HPC" if HPC_MODE else "Colab"
print(f"Environment: {ENV} | CV workers: {N_CORES} | BLAS/OpenMP threads per worker: {os.environ['OMP_NUM_THREADS']}")




In [ ]:
# !pip install rdkit optuna shap crepes -q




In [ ]:
import pandas as pd
import numpy as np
import json
import os
import hashlib
import datetime
import platform
import subprocess
import copy
import time
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from venn_abers import VennAbers
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    average_precision_score, brier_score_loss, r2_score, mean_squared_error,
    mean_absolute_error,
)

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

from scipy.stats import spearmanr, wilcoxon
import joblib
import shap

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.rdBase import BlockLogs

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.ERROR)

import crepes
from crepes import WrapClassifier, WrapRegressor

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

print('Imports complete.')





In [ ]:
from pathlib import Path

if HPC_MODE:
    PROJECT_DIR = Path("./")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/My Drive/gpcr_benchmark")

for subdir in ["data/processed", "ml/features", "ml/models", "ml/results", "logs"]:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

processed_path = PROJECT_DIR / "data" / "processed"
features_path  = PROJECT_DIR / "ml" / "features"
models_path    = PROJECT_DIR / "ml" / "models"
results_path   = PROJECT_DIR / "ml" / "results"
logs_path      = PROJECT_DIR / "logs"

print(f"Project directory: {PROJECT_DIR}")





In [ ]:
# =============================================================================
# GLOBAL CONFIGURATION — single source of truth for every combination this
# notebook runs. No per-target/per-pool branching anywhere downstream reads
# from anything but this cell.
# =============================================================================

RANDOM_STATE = 42
ACTIVITY_THRESHOLD = 6.0  # must match notebook 02's ACTIVITY_THRESHOLD exactly

TARGETS = ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']
POOLS = ['ki', 'ki_ic50', 'full']
POOL_LABELS = {'ki': 'Ki', 'ki_ic50': 'Ki + IC50', 'full': 'Ki + IC50 + EC50'}

# Exactly 3 feature representations — explicitly bounded scope (Q3), not an
# exhaustive fingerprint-representation study.
FEATURE_REPS = ['morgan', 'descriptors', 'combined']
FEATURE_REP_LABELS = {
    'morgan': 'Morgan fingerprints',
    'descriptors': 'Physicochemical descriptors',
    'combined': 'Morgan + descriptors',
}

# Exactly 3 algorithms, classification AND regression variants of each.
ALGORITHMS = ['rf', 'xgb', 'lgb']
ALGO_LABELS = {'rf': 'Random Forest', 'xgb': 'XGBoost', 'lgb': 'LightGBM'}
TASKS = ['classification', 'regression']

# Morgan fingerprint parameters — verified against the predecessor notebook's
# `generate_fingerprints(method="Morgan", radius=2, n_bits=2048)` default and
# its `fp_sizes: {"Morgan": 2048}` config entry.
MORGAN_RADIUS = 2
MORGAN_NBITS = 2048

# The 10 physicochemical descriptors notebook 02 already computed — loaded
# from the cleaned CSVs, never recomputed here (compute-once-reuse-everywhere).
# Exact RDKit calls (verified against notebook 02 Step 7, `annotate_descriptors`):
#   MW=Descriptors.MolWt, LogP=Crippen.MolLogP, TPSA=Descriptors.TPSA,
#   HBD=Lipinski.NumHDonors, HBA=Lipinski.NumHAcceptors,
#   RotBonds=Descriptors.NumRotatableBonds, HeavyAtomCount=Descriptors.HeavyAtomCount,
#   RingCount=Descriptors.RingCount, AromaticRingCount=Descriptors.NumAromaticRings,
#   FractionCSP3=Descriptors.FractionCSP3
DESCRIPTOR_COLS = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds',
                   'HeavyAtomCount', 'RingCount', 'AromaticRingCount', 'FractionCSP3']

# Hyperparameter policy: equal Optuna tuning budget
# across all 5 targets — 50 trials per (target, pool, model, task), tuned
# ONCE per (target, pool) on the 'combined' representation and reused across
# all 3 feature representations (see Module D for the rationale). No
# fixed-hyperparameter fallback path exists anywhere below this cell.
N_OPTUNA_TRIALS = 50
OPTUNA_CV_FOLDS = 5   # scaffold GroupKFold folds INSIDE each Optuna trial

# HPC parallelism policy: parallelize ACROSS Optuna trials and keep each trial's
# inner CV bounded. Models inside each CV fold keep n_jobs=1, and BLAS/OpenMP
# threads are fixed to 1 in the first cell, so this avoids nested oversubscription.
# Capped at 2 concurrent trials (not 4): each concurrent trial is a separate
# writer to the same per-combo Optuna SQLite study file, and on Lustre (a
# parallel filesystem with weak file-locking support) 4 concurrent writers
# reliably produced "database is locked" errors that killed multi-hour runs
# even with an extended busy-timeout. 2 writers is materially more reliable
# in practice; the lost throughput is preferable to losing a partially
# completed multi-hour run.
OPTUNA_PARALLEL_TRIALS = max(1, min(2, N_CORES // max(1, OPTUNA_CV_FOLDS)))
OPTUNA_CV_N_JOBS = max(1, min(OPTUNA_CV_FOLDS, N_CORES // OPTUNA_PARALLEL_TRIALS))

# -----------------------------------------------------------------------------
# TUNING MODE — 'optuna' (default, the locked primary policy: equal 50-trial
# Optuna budget tuned leak-free inside each fold) OR 'fixed' (no tuning: every
# model uses the documented FIXED_HYPERPARAMS below, identically across all
# targets). 'fixed' is a legitimate, fully leak-free alternative — with no
# hyperparameter SELECTION there is nothing to leak — and is literally the
# "fixed-identical hyperparameters across all targets" policy (the other option
# originally weighed against equal-Optuna-budget). Use it for a fast run, a
# compute-limited environment, or a study that deliberately fixes hyperparameters;
# disclose it in Methods as fixed-hyperparameter, not as "tuning we skipped".
# Everything downstream (nested CV, deployment, best-algorithm selection by
# inner-CV score, SHAP, etc.) works identically in both modes.
TUNING_MODE = 'optuna'
assert TUNING_MODE in ('optuna', 'fixed')

# Fixed hyperparameters used when TUNING_MODE == 'fixed'. Class imbalance IS
# handled (class_weight for RF/LightGBM; scale_pos_weight for XGBoost is set to
# the empirical neg/pos ratio of the fitting subset at fit time — see
# tune_on_subset's fixed branch) so this does NOT repeat the predecessor bug
# where a fixed-default path silently dropped class weighting. Every value here
# is disclosed in the run manifest.
FIXED_HYPERPARAMS = {
    'rf_classification':  {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 2,
                            'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True,
                            'class_weight': 'balanced'},
    'xgb_classification': {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05,
                            'subsample': 0.8, 'colsample_bytree': 0.3, 'colsample_bylevel': 0.7,
                            'min_child_weight': 3, 'gamma': 0.1, 'reg_alpha': 0.5, 'reg_lambda': 1.0},
    'lgb_classification': {'n_estimators': 400, 'num_leaves': 31, 'max_depth': 12,
                            'learning_rate': 0.05, 'subsample': 0.8, 'subsample_freq': 1,
                            'colsample_bytree': 0.3, 'min_child_samples': 20,
                            'reg_alpha': 0.5, 'reg_lambda': 1.0, 'class_weight': 'balanced'},
    'rf_regression':      {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 2,
                            'min_samples_leaf': 1, 'max_features': 0.5},
    'xgb_regression':     {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05,
                            'subsample': 0.8, 'colsample_bytree': 0.3, 'colsample_bylevel': 0.7,
                            'min_child_weight': 3, 'gamma': 0.1, 'reg_alpha': 0.5, 'reg_lambda': 1.0},
    'lgb_regression':     {'n_estimators': 400, 'num_leaves': 31, 'max_depth': 12,
                            'learning_rate': 0.05, 'subsample': 0.8, 'subsample_freq': 1,
                            'colsample_bytree': 0.3, 'min_child_samples': 20,
                            'reg_alpha': 0.5, 'reg_lambda': 1.0},
}

# Repeated scaffold-split validation count. Quoted exactly from the
# predecessor's `get_config()`: 'repeated_scaffold_split': {'n_repeats': 20 if
# is_cb2 else 10} — CB2 (the primary/full-rigor target) used 20. That number
# is now applied uniformly to all 5 targets x 3 pools (no more primary/
# secondary asymmetry).
N_REPEATS = 20        # repeated scaffold splits used ONLY for descriptive
                      # error analysis (Module H persistent-misclassification),
                      # NOT for the headline performance estimate.
N_OUTER_FOLDS = 5     # outer folds for the leak-free nested-CV benchmark
TEST_SIZE = 0.2
Y_RAND_ITERATIONS = 100

# Okabe-Ito colorblind-safe palette, one color per target (matches notebook 02).
TARGET_COLORS = dict(zip(TARGETS, ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7']))
ALGO_COLORS = {'rf': '#0072B2', 'xgb': '#D55E00', 'lgb': '#009E73', 'ensemble': '#000000'}

assert 0.0 < ACTIVITY_THRESHOLD < 14.0
assert all(tk in ('drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5') for tk in TARGETS)
assert set(FEATURE_REPS) == {'morgan', 'descriptors', 'combined'}
assert set(ALGORITHMS) == {'rf', 'xgb', 'lgb'}

# Optional PBS sharding: set GPCR_COMBOS to a comma-separated list such as
# "drd2_ki,drd2_ki_ic50,drd2_full". Each shard owns only its requested
# (target, pool) combinations, which is safe because per-combo outputs live in
# separate ml/results/{target}_{pool}/ and ml/models/{target}_{pool}/ folders.
ALL_COMBOS = [(target, pool) for target in TARGETS for pool in POOLS]
_requested_combos_raw = os.environ.get('GPCR_COMBOS', '').strip()
if _requested_combos_raw:
    _requested_combos = {x.strip() for x in _requested_combos_raw.split(',') if x.strip()}
    _valid_combos = {f'{target}_{pool}' for target, pool in ALL_COMBOS}
    _unknown_combos = sorted(_requested_combos - _valid_combos)
    if _unknown_combos:
        raise ValueError(f'Unknown GPCR_COMBOS entries: {_unknown_combos}; valid={sorted(_valid_combos)}')
    RUN_COMBOS = [(target, pool) for target, pool in ALL_COMBOS if f'{target}_{pool}' in _requested_combos]
else:
    RUN_COMBOS = ALL_COMBOS
RUN_TARGETS = sorted({target for target, _ in RUN_COMBOS}, key=TARGETS.index)
RUN_POOLS = sorted({pool for _, pool in RUN_COMBOS}, key=POOLS.index)
IS_SHARDED_RUN = bool(_requested_combos_raw)
SHARD_TAG = 'all' if not IS_SHARDED_RUN else '_'.join(f'{target}_{pool}' for target, pool in RUN_COMBOS)

def shard_name(filename):
    '''Avoid concurrent shard jobs clobbering global summary outputs.
    Per-combo outputs remain in ml/results/{target}_{pool}/ and are shared
    intentionally; only cross-combo/global outputs get a shard suffix.'''
    if not IS_SHARDED_RUN:
        return filename
    p = Path(filename)
    return f'{p.stem}_{SHARD_TAG}{p.suffix}'

print(f'Targets                : {TARGETS}')
print(f'Pools                  : {POOLS}')
print(f'Feature representations: {FEATURE_REPS}')
print(f'Algorithms             : {ALGORITHMS}')
print(f'Optuna trials          : {N_OPTUNA_TRIALS} per (target, pool, model, task)')
print(f'Optuna parallelism     : {OPTUNA_PARALLEL_TRIALS} trials x {OPTUNA_CV_N_JOBS} CV workers ~= {OPTUNA_PARALLEL_TRIALS * OPTUNA_CV_N_JOBS} active workers')
print(f'Repeated scaffold splits: {N_REPEATS} per (target, pool, representation)')
print(f'Total combinations     : {len(TARGETS)} targets x {len(POOLS)} pools = {len(TARGETS)*len(POOLS)}')
print(f'Run combinations       : {[f"{t}_{p}" for t, p in RUN_COMBOS]}')








In [ ]:
def get_config(target, pool):
    '''Central per-(target, pool) path/threshold configuration. No
    analysis function below hardcodes a target-specific value — everything
    reads from this dict.'''
    assert target in TARGETS, f'Unknown target: {target!r}'
    assert pool in POOLS, f'Unknown pool: {pool!r}'
    combo_key = f'{target}_{pool}'
    return {
        'target': target,
        'pool': pool,
        'combo_key': combo_key,
        'cleaned_data_path': processed_path / f'cleaned_data_{target}_{pool}.csv',
        'features_dir': features_path / combo_key,
        'models_dir': models_path / combo_key,
        'results_dir': results_path / combo_key,
        'random_state': RANDOM_STATE,
        'test_size': TEST_SIZE,
        'n_optuna_trials': N_OPTUNA_TRIALS,
        'optuna_cv_folds': OPTUNA_CV_FOLDS,
        'n_repeats': N_REPEATS,
    }

def validate_config(cfg):
    errors = []
    if not cfg['cleaned_data_path'].exists():
        errors.append(f"cleaned_data_path does not exist: {cfg['cleaned_data_path']}")
    if not (0.0 < cfg['test_size'] < 1.0):
        errors.append(f"test_size out of range: {cfg['test_size']}")
    if cfg['n_optuna_trials'] < 1:
        errors.append(f"n_optuna_trials must be >= 1")
    if errors:
        raise ValueError('Config validation failed:\n  - ' + '\n  - '.join(errors))
    return True

print('get_config() and validate_config() defined.')





In [ ]:
# =============================================================================
# PUBLICATION FIGURE STYLE — identical convention to notebook 02: no
# on-figure titles (caption carries that), no code identifiers in any
# label/legend, American spelling, 300dpi PNG + vector PDF, panel letters
# via panel_label() instead of descriptive titles.
# =============================================================================
mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def combo_figures_dir(cfg):
    d = cfg['results_dir'] / 'figures'
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_fig(fig, stem, out_dir):
    for ext in ("png", "pdf"):
        fig.savefig(Path(out_dir) / f"{stem}.{ext}")
    plt.close(fig)

def panel_label(ax, letter):
    '''Bold panel letter (A/B/C...), top-left — NOT a descriptive title.'''
    ax.set_title(letter, loc='left', fontweight='bold', fontsize=12)

all_outputs = {}

def _register(name, path_obj, n_rows=None):
    entry = {'path': str(path_obj)}
    if n_rows is not None:
        entry['n_rows'] = n_rows
    all_outputs[name] = entry

print('Figure style configured.')





In [ ]:
def _hash_file(path, chunk_size=1 << 20):
    '''SHA-256 of a file, chunked so large CSVs don't blow up memory.'''
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def _capture_package_versions():
    '''Enumerate __version__ for every package THIS notebook actually
    imports — programmatic, not a fixed hardcoded list (lesson 0b).'''
    packages = ['rdkit', 'numpy', 'pandas', 'sklearn', 'xgboost', 'lightgbm', 'venn_abers',
                'optuna', 'shap', 'joblib', 'crepes', 'scipy', 'matplotlib']
    versions = {}
    for pkg in packages:
        try:
            mod = __import__(pkg)
            versions[pkg] = getattr(mod, '__version__', 'unknown')
        except ImportError:
            pass
    return versions


def write_manifest(manifest_path, config_summary, outputs, inputs=None):
    '''Single source-of-truth record of config + inputs + outputs +
    timestamp (+ best-effort git commit, Python version, SHA-256 of every
    input AND output file) — identical convention to notebooks 01/02.'''
    git_hash = None
    try:
        git_hash = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], stderr=subprocess.DEVNULL, cwd=str(PROJECT_DIR)
        ).decode().strip()
    except Exception:
        pass

    def _hash_registry(registry):
        hashed = {}
        for name, meta in registry.items():
            meta = dict(meta)
            p = Path(meta.get('path', ''))
            if p.exists() and p.is_file():
                try:
                    meta['sha256'] = _hash_file(p)
                except Exception:
                    pass
            hashed[name] = meta
        return hashed

    manifest = {
        'timestamp': datetime.datetime.now().isoformat(),
        'python_version': platform.python_version(),
        'git_commit': git_hash,
        'config': config_summary,
        'inputs': _hash_registry(inputs) if inputs else {},
        'outputs': _hash_registry(outputs),
        'package_versions': _capture_package_versions(),
    }
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2, default=str)
    print(f'Manifest saved: {manifest_path} ({len(manifest["inputs"])} inputs, {len(manifest["outputs"])} outputs hashed)')
    return manifest


def log_milestone(target, pool, message):
    '''Append a timestamped line to a run log — survives a disconnect
    that would otherwise lose print-only progress.'''
    log_path = logs_path / f'run_log_03_ml_benchmark_{target}_{pool}.txt'
    with open(log_path, 'a') as f:
        f.write(f'[{datetime.datetime.now().isoformat()}] {message}\n')
    return log_path

print('Manifest/logging helpers defined.')





In [ ]:
# ---- Environment snapshot for reproducibility: freeze the exact installed
# package set to a file (an auto-generated environment record), and print the
# key ML-stack versions. Exact versions are also captured in the run manifest;
# this file is the human-readable companion the Methods section cites. ----
try:
    frozen = subprocess.check_output(['pip', 'freeze'], stderr=subprocess.DEVNULL).decode()
    env_snapshot_path = logs_path / 'environment_frozen_03_ml_benchmark.txt'
    with open(env_snapshot_path, 'w') as f:
        f.write(frozen)
    print(f'Environment snapshot written: {env_snapshot_path}')
except Exception as e:
    print(f'Could not write pip freeze snapshot ({e}) — manifest still captures versions.')

_key_versions = _capture_package_versions()
print('Key package versions:')
for pkg, ver in _key_versions.items():
    print(f'  {pkg:12s} {ver}')





## Module A: Load and Validate

Load notebook 02's frozen outputs for all 5 targets x 3 pools, re-hash every
file against `manifest_02_data_cleaning.json` (not just trusted by
filename), merge in the precomputed scaffold assignments, and run a battery
of assertions before anything downstream is allowed to train on this data.
Mirrors notebook 02's own "assertions: fail fast" pattern — if any check
here fails, this notebook raises rather than silently training on
data that failed validation.


In [ ]:
# ---- Load notebook 02's manifest and re-hash cleaned data files against it ----
manifest_02_path = processed_path / 'manifest_02_data_cleaning.json'
assert manifest_02_path.exists(), f'notebook 02 manifest not found: {manifest_02_path}'

with open(manifest_02_path) as f:
    manifest_02 = json.load(f)

hash_mismatches = []
for target, pool in RUN_COMBOS:
        fname = f'cleaned_data_{target}_{pool}.csv'
        fpath = processed_path / fname
        assert fpath.exists(), f'Missing required input: {fpath}'
        recorded = manifest_02['outputs'].get(fname, {})
        recorded_hash = recorded.get('sha256')
        actual_hash = _hash_file(fpath)
        if recorded_hash is not None and recorded_hash != actual_hash:
            hash_mismatches.append(fname)

if hash_mismatches:
    raise AssertionError(
        f'{len(hash_mismatches)} cleaned_data file(s) do not match the SHA-256 '
        f'recorded in manifest_02_data_cleaning.json — these files were modified '
        f'or regenerated since notebook 02 last ran: {hash_mismatches}'
    )
print(f'All {len(TARGETS)*len(POOLS)} cleaned_data files hash-verified against manifest_02_data_cleaning.json.')





In [ ]:
# ---- Load cleaned datasets for the combinations in scope for this run ----
# Full run loads all 15 (target, pool) combinations; a sharded run loads only
# the combinations named in RUN_COMBOS for this job.
raw_cleaned = {}
for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        validate_config(cfg)
        df = pd.read_csv(cfg['cleaned_data_path'])
        raw_cleaned[(target, pool)] = df
        print(f'{target:8s} / {pool:8s} : {len(df):6d} compounds, {df.shape[1]} columns')

n_combos = len(raw_cleaned)
assert n_combos == len(RUN_COMBOS), f'Expected {len(RUN_COMBOS)} (target, pool) combinations, got {n_combos}'





In [ ]:
# ---- Load shared reference tables (scaffold assignments, summaries) ----
scaffold_assignments = pd.read_csv(processed_path / 'compound_scaffold_assignments.csv')
dataset_summary = pd.read_csv(processed_path / 'dataset_summary_all_targets_pools.csv')
scaffold_diversity_summary = pd.read_csv(processed_path / 'scaffold_diversity_summary.csv')

required_scaffold_cols = {'target', 'activity_pool', 'molecule_chembl_id',
                           'global_scaffold_id', 'within_dataset_scaffold_rank_id',
                           'scaffold_size', 'is_singleton_scaffold'}
missing_cols = required_scaffold_cols - set(scaffold_assignments.columns)
assert not missing_cols, f'compound_scaffold_assignments.csv missing columns: {missing_cols}'

print(f'compound_scaffold_assignments.csv : {len(scaffold_assignments)} rows')
print(f'dataset_summary_all_targets_pools : {len(dataset_summary)} rows')
print(f'scaffold_diversity_summary        : {len(scaffold_diversity_summary)} rows')





In [ ]:
# ---- Merge scaffold assignments into each cleaned dataset; compute
# global_compound_id using notebook 02's exact convention (SHA256 of
# clean_smiles, first 12 hex chars, 'CMPD_' prefix) so IDs are consistent
# across notebooks without needing the column to already exist in the
# cleaned CSVs. ----
def add_global_compound_id(df):
    df = df.copy()
    df['global_compound_id'] = df['clean_smiles'].apply(
        lambda s: 'CMPD_' + hashlib.sha256(str(s).encode('utf-8')).hexdigest()[:12])
    return df

merged = {}
merge_report = []
for (target, pool), df in raw_cleaned.items():
    df = add_global_compound_id(df)
    scaf_sub = scaffold_assignments[
        (scaffold_assignments['target'] == target) & (scaffold_assignments['activity_pool'] == pool)
    ][['molecule_chembl_id', 'global_scaffold_id', 'within_dataset_scaffold_rank_id',
       'scaffold_size', 'is_singleton_scaffold']]
    n_before = len(df)
    df = df.merge(scaf_sub, on='molecule_chembl_id', how='left', validate='one_to_one')
    n_unmatched = df['global_scaffold_id'].isna().sum()
    merge_report.append({'target': target, 'pool': pool, 'n_compounds': n_before,
                         'n_unmatched_scaffold': int(n_unmatched)})
    merged[(target, pool)] = df

merge_report_df = pd.DataFrame(merge_report)
print(merge_report_df.to_string(index=False))
assert (merge_report_df['n_unmatched_scaffold'] == 0).all(), (
    'Some compounds failed to match a scaffold assignment — merge key '
    '(target, activity_pool, molecule_chembl_id) is not a 1:1 join.'
)





In [ ]:
# ---- Validation battery: activity threshold consistency, scaffold-ID
# completeness, descriptor completeness. Raise on any failure — no model
# trains on data that failed validation. ----
validation_rows = []
validation_failures = []

for (target, pool), df in merged.items():
    row = {'target': target, 'pool': pool, 'n_compounds': len(df)}

    # Activity threshold consistency: activity==1 iff pActivity >= ACTIVITY_THRESHOLD
    expected_activity = (df['pActivity'] >= ACTIVITY_THRESHOLD).astype(int)
    n_activity_mismatch = int((expected_activity != df['activity']).sum())
    row['n_activity_threshold_mismatch'] = n_activity_mismatch
    if n_activity_mismatch > 0:
        validation_failures.append(f'{target}/{pool}: {n_activity_mismatch} activity-label mismatches')

    # Scaffold IDs present and non-null
    n_null_scaffold = int(df['global_scaffold_id'].isna().sum())
    row['n_null_scaffold_id'] = n_null_scaffold
    if n_null_scaffold > 0:
        validation_failures.append(f'{target}/{pool}: {n_null_scaffold} null global_scaffold_id')

    # Descriptor completeness — no unexpected NaNs
    n_desc_nan = int(df[DESCRIPTOR_COLS].isna().sum().sum())
    row['n_descriptor_nan'] = n_desc_nan
    if n_desc_nan > 0:
        validation_failures.append(f'{target}/{pool}: {n_desc_nan} NaN values across descriptor columns')

    # global_compound_id uniqueness within this dataset
    n_dup_compound_id = int(df['global_compound_id'].duplicated().sum())
    row['n_duplicate_compound_id'] = n_dup_compound_id
    if n_dup_compound_id > 0:
        validation_failures.append(f'{target}/{pool}: {n_dup_compound_id} duplicate global_compound_id')

    validation_rows.append(row)

validation_df = pd.DataFrame(validation_rows)
print(validation_df.to_string(index=False))

validation_report_path = processed_path / shard_name('nb03_module_a_validation_report.csv')
validation_df.to_csv(validation_report_path, index=False)
_register('nb03_module_a_validation_report.csv', validation_report_path, len(validation_df))

if validation_failures:
    raise AssertionError('Module A validation failed:\n  - ' + '\n  - '.join(validation_failures))
print(f'\nAll validation checks passed for all {len(RUN_COMBOS)} (target, pool) combination(s) in this run.')

# Disclosed exception (not a bug): CCR5's 'ki'-only pool is the
# thinnest dataset in the panel -- flagged here explicitly, not treated as
# a bug. It is still cleaned, featurized, and modeled like every other
# combination; Module D's activity-pool sensitivity figure excludes it
# specifically (see that cell for the disclosed rationale).
ccr5_ki_n = validation_df.loc[(validation_df['target'] == 'ccr5') & (validation_df['pool'] == 'ki'), 'n_compounds']
if len(ccr5_ki_n) > 0:
    msg = (
        f'Disclosed exception: CCR5/ki has {int(ccr5_ki_n.iloc[0])} compounds -- the thinnest '
        f'combination in the panel. Trained identically to every other '
        f'combination below; excluded only from the activity-pool sensitivity comparison figure '
        f'in Module D, where its small-sample noise would be misread as a real pool effect.'
    )
    print('\n' + msg)





## Module B: Feature Engineering

Generate Morgan fingerprints (radius 2, 2048 bits), load the physicochemical
descriptors already computed in notebook 02, and build the combined
representation — each generated exactly ONCE per (target, pool) and
persisted to `ml/features/{target}_{pool}/`, never regenerated mid-pipeline
(compute-once-reuse-everywhere).


In [ ]:
def smiles_to_mols(smiles_list):
    '''RDKit mol parsing with a BlockLogs() context so per-molecule
    warnings don't flood output across 15 x thousands of compounds.'''
    mols = []
    with BlockLogs():
        for smi in smiles_list:
            mols.append(Chem.MolFromSmiles(smi) if isinstance(smi, str) else None)
    return mols


def compute_morgan_matrix(mols, radius=MORGAN_RADIUS, n_bits=MORGAN_NBITS):
    '''Morgan (ECFP-equivalent) fingerprint matrix, radius=2 nBits=2048 —
    verified against the predecessor notebook's fp_sizes config and its
    generate_fingerprints() default.'''
    fps = np.zeros((len(mols), n_bits), dtype=np.uint8)
    for i, mol in enumerate(mols):
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
        arr = np.zeros(n_bits, dtype=np.uint8)
        for on_bit in fp.GetOnBits():
            arr[on_bit] = 1
        fps[i] = arr
    return fps


def morgan_feature_names(n_bits=MORGAN_NBITS):
    return [f'Morgan_{i}' for i in range(n_bits)]

print('Feature-generation functions defined.')





In [ ]:
# ---- Generate + persist all 3 feature representations for all 15 combos ----
feature_manifest_rows = []

for (target, pool), df in merged.items():
    cfg = get_config(target, pool)
    cfg['features_dir'].mkdir(parents=True, exist_ok=True)

    mols = smiles_to_mols(df['clean_smiles'].tolist())
    n_invalid = sum(m is None for m in mols)
    if n_invalid > 0:
        print(f'  ⚠️  {target}/{pool}: {n_invalid} SMILES failed to parse — corresponding rows get zero-vector Morgan features')

    X_morgan = compute_morgan_matrix(mols)
    X_desc = df[DESCRIPTOR_COLS].to_numpy(dtype=float)
    X_desc = np.nan_to_num(X_desc, nan=0.0, posinf=0.0, neginf=0.0)
    X_combined = np.hstack([X_desc, X_morgan])

    feature_arrays = {'morgan': X_morgan, 'descriptors': X_desc, 'combined': X_combined}
    feature_names = {
        'morgan': morgan_feature_names(),
        'descriptors': list(DESCRIPTOR_COLS),
        'combined': list(DESCRIPTOR_COLS) + morgan_feature_names(),
    }

    for rep in FEATURE_REPS:
        arr_path = cfg['features_dir'] / f'{rep}.npy'
        np.save(arr_path, feature_arrays[rep])
        names_path = cfg['features_dir'] / f'{rep}_feature_names.json'
        with open(names_path, 'w') as f:
            json.dump(feature_names[rep], f)
        feature_manifest_rows.append({
            'target': target, 'pool': pool, 'representation': rep,
            'n_compounds': feature_arrays[rep].shape[0],
            'n_features': feature_arrays[rep].shape[1],
            'path': str(arr_path),
        })

    # Persist labels + IDs alongside features so downstream modules never
    # need to re-derive them from the cleaned CSV.
    labels_df = df[['global_compound_id', 'molecule_chembl_id', 'global_scaffold_id',
                     'scaffold_size', 'is_singleton_scaffold', 'activity', 'pActivity']].copy()
    labels_df.to_csv(cfg['features_dir'] / 'labels.csv', index=False)

    print(f'{target:8s}/{pool:8s}: morgan={X_morgan.shape} descriptors={X_desc.shape} combined={X_combined.shape}')

feature_manifest_df = pd.DataFrame(feature_manifest_rows)
feature_manifest_out = processed_path / 'feature_generation_manifest.csv'
feature_manifest_df.to_csv(feature_manifest_out, index=False)
_register('feature_generation_manifest.csv', feature_manifest_out, len(feature_manifest_df))
print(f'\nFeature generation manifest saved: {feature_manifest_out} ({len(feature_manifest_df)} rows)')





## Module C: Dataset Diagnostics

Feature variance, missingness/infinite-value checks, correlation structure,
and PCA — for all 3 feature representations, all 15 combinations. Builds on
notebook 02's existing descriptor-only chemical-space PCA (which covered the
`full` pool across all 5 targets) by adding a Morgan-fingerprint and
combined-feature PCA for comparison — kept proportionate (one comparison
figure), not a large new sub-study.


In [ ]:
def near_zero_variance_report(X, feature_names, threshold=1e-4):
    '''Flag (not drop) near-zero-variance features — reported for
    transparency, the modeling pipeline does not silently remove features.'''
    variances = np.var(X.astype(float), axis=0)
    flagged = variances < threshold
    return pd.DataFrame({
        'feature': feature_names,
        'variance': variances,
        'near_zero_variance': flagged,
    })


def missingness_report(X):
    X = X.astype(float)
    return {
        'n_nan': int(np.isnan(X).sum()),
        'n_inf': int(np.isinf(X).sum()),
        'n_total': int(X.size),
    }

print('Diagnostic helper functions defined.')





In [ ]:
# ---- Run diagnostics for all 15 combos x 3 representations ----
diagnostics_rows = []
nzv_records = []
correlation_flag_records = []

for (target, pool) in merged.keys():
    cfg = get_config(target, pool)
    for rep in FEATURE_REPS:
        X = np.load(cfg['features_dir'] / f'{rep}.npy')
        with open(cfg['features_dir'] / f'{rep}_feature_names.json') as f:
            fnames = json.load(f)

        miss = missingness_report(X)
        nzv_df = near_zero_variance_report(X, fnames)
        n_flagged = int(nzv_df['near_zero_variance'].sum())

        # Feature sparsity/density — cheap, useful supplementary characterisation
        # (Morgan fingerprints are mostly-zero bit vectors; descriptors are dense).
        Xf = X.astype(float)
        n_zero = int((Xf == 0).sum())
        pct_zero = n_zero / Xf.size * 100 if Xf.size else 0.0

        diagnostics_rows.append({
            'target': target, 'pool': pool, 'representation': rep,
            'n_features': X.shape[1], 'n_compounds': X.shape[0],
            **miss, 'n_near_zero_variance_features': n_flagged,
            'pct_zero_entries': pct_zero, 'density': 100.0 - pct_zero,
        })

        if rep == 'descriptors':
            # Only descriptors are low-dimensional enough for a full
            # pairwise Spearman correlation matrix to be meaningful/cheap.
            corr = pd.DataFrame(X, columns=fnames).corr(method='spearman')
            high_corr_pairs = []
            for i in range(len(fnames)):
                for j in range(i + 1, len(fnames)):
                    r = corr.iloc[i, j]
                    if abs(r) >= 0.8:
                        high_corr_pairs.append({
                            'target': target, 'pool': pool,
                            'feature_a': fnames[i], 'feature_b': fnames[j],
                            'spearman_r': float(r),
                        })
            correlation_flag_records.extend(high_corr_pairs)

        nzv_df.insert(0, 'representation', rep)
        nzv_df.insert(0, 'pool', pool)
        nzv_df.insert(0, 'target', target)
        # Only persist the flagged rows for the high-dimensional Morgan/
        # combined representations (2048+ rows per combo x 15 combos would
        # otherwise bloat the CSV with all-zero entries that carry no signal).
        nzv_records.append(nzv_df[nzv_df['near_zero_variance']] if rep != 'descriptors' else nzv_df)

diagnostics_df = pd.DataFrame(diagnostics_rows)
nzv_all_df = pd.concat(nzv_records, ignore_index=True)
correlation_flags_df = pd.DataFrame(correlation_flag_records)

diag_path = processed_path / shard_name('feature_diagnostics_summary.csv')
nzv_path = processed_path / shard_name('near_zero_variance_features.csv')
corr_path = processed_path / shard_name('descriptor_high_correlation_pairs.csv')
diagnostics_df.to_csv(diag_path, index=False)
nzv_all_df.to_csv(nzv_path, index=False)
correlation_flags_df.to_csv(corr_path, index=False)
_register('feature_diagnostics_summary.csv', diag_path, len(diagnostics_df))
_register('near_zero_variance_features.csv', nzv_path, len(nzv_all_df))
_register('descriptor_high_correlation_pairs.csv', corr_path, len(correlation_flags_df))

print(diagnostics_df.to_string(index=False))
print(f'\nHigh-correlation descriptor pairs (|Spearman r| >= 0.8): {len(correlation_flags_df)}')
if len(correlation_flags_df) > 0:
    print(correlation_flags_df.to_string(index=False))





In [ ]:
# ---- PCA comparison: descriptors (notebook 02, reused) vs Morgan vs
# combined, full pool, all 5 targets — proportionate addition, one figure.
# Cross-target PCA requires all full-pool feature matrices, so shard jobs skip
# it; the final unsharded aggregation pass regenerates the complete output. ----
if IS_SHARDED_RUN:
    print(f'Skipping cross-target PCA representation comparison in shard run: {SHARD_TAG}')
else:
    existing_pca_variance = pd.read_csv(processed_path / 'chemical_space_pca_variance.csv')
    print('Notebook 02 descriptor-only PCA (full pool, all targets), reused here:')
    print(existing_pca_variance.to_string(index=False))

    pca_comparison_rows = []
    pca_comparison_rows.append({
        'representation': 'descriptors', 'pc1_variance_ratio': float(existing_pca_variance.iloc[0]['explained_variance_ratio']),
        'pc2_variance_ratio': float(existing_pca_variance.iloc[1]['explained_variance_ratio']),
    })

    for rep, reducer_name in [('morgan', 'TruncatedSVD'), ('combined', 'TruncatedSVD')]:
        frames = []
        for target in TARGETS:
            cfg = get_config(target, 'full')
            X = np.load(cfg['features_dir'] / f'{rep}.npy').astype(float)
            frames.append(X)
        X_all = np.vstack(frames)
        # Morgan bits are binary/sparse-ish — TruncatedSVD (no centering) is the
        # standard choice over PCA for this kind of high-dimensional binary matrix.
        reducer = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
        reducer.fit(X_all)
        pca_comparison_rows.append({
            'representation': rep,
            'pc1_variance_ratio': float(reducer.explained_variance_ratio_[0]),
            'pc2_variance_ratio': float(reducer.explained_variance_ratio_[1]),
        })

    pca_comparison_df = pd.DataFrame(pca_comparison_rows)
    pca_comparison_path = processed_path / shard_name('chemical_space_pca_representation_comparison.csv')
    pca_comparison_df.to_csv(pca_comparison_path, index=False)
    _register('chemical_space_pca_representation_comparison.csv', pca_comparison_path, len(pca_comparison_df))
    print('\nExplained variance (top-2 components) by feature representation, full pool, all targets:')
    print(pca_comparison_df.to_string(index=False))






In [ ]:
# ---- Figure: explained variance by feature representation ----
fig_dir = processed_path / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

if IS_SHARDED_RUN:
    print(f'Skipping PCA representation figure in shard run: {SHARD_TAG}')
else:
    fig, ax = plt.subplots(figsize=(5, 4))
    x = np.arange(len(pca_comparison_df))
    width = 0.35
    ax.bar(x - width / 2, pca_comparison_df['pc1_variance_ratio'] * 100, width, label='Component 1', color='#0072B2')
    ax.bar(x + width / 2, pca_comparison_df['pc2_variance_ratio'] * 100, width, label='Component 2', color='#E69F00')
    ax.set_xticks(x)
    ax.set_xticklabels([FEATURE_REP_LABELS[r] for r in pca_comparison_df['representation']], rotation=20, ha='right')
    ax.set_ylabel('Explained variance (%)')
    ax.legend(frameon=False)
    panel_label(ax, 'A')
    fig.tight_layout()
    save_fig(fig, 'figure_pca_variance_by_representation', fig_dir)
    _register('figure_pca_variance_by_representation.png', fig_dir / 'figure_pca_variance_by_representation.png')
    _register('figure_pca_variance_by_representation.pdf', fig_dir / 'figure_pca_variance_by_representation.pdf')
    print('Figure saved: figure_pca_variance_by_representation (plot-source data already in chemical_space_pca_representation_comparison.csv)')



## Module D: Benchmarking (leak-free nested cross-validation)

The core of this notebook, and the source of the headline generalization
numbers. For every (target x pool):

**Nested scaffold cross-validation** — an outer GroupKFold over scaffold
groups (N_OUTER_FOLDS folds; whole scaffolds held out per fold). Inside each
outer fold, hyperparameters are tuned on the OUTER-TRAIN fold only (inner
scaffold GroupKFold, 50 Optuna trials on the combined representation), then
every algorithm x representation is trained on outer-train and evaluated ONCE
on the untouched outer-test. No outer-test compound ever influences
hyperparameter selection, so the distribution of outer-test scores is an
unbiased (leak-free) estimate of cross-target generalization, and the
RF/XGBoost/LightGBM comparison happens on outer-test with no prior algorithm
selection on that same test.

**Deployment tuning** — separately, hyperparameters for the single
deployed/calibrated/explained model used in Modules E/F/H are tuned on the
canonical `cal_train` partition ONLY, so that model's canonical test set is
never seen during its own tuning either. The best algorithm for those
single-model analyses is chosen by deployment inner-CV score (not by any test
performance).

**Why tune once on 'combined' per fold and reuse across the 3 feature
representations:** the ablation (Q3 — which features transfer across GPCRs)
asks how much the INPUT FEATURE SET matters holding everything else fixed.
Re-tuning separately per representation would confound the feature effect with
a re-discovered-hyperparameter effect; holding hyperparameters fixed across
the representation comparison (within each fold) isolates the variable the
ablation is meant to measure. This is a deliberate controlled-ablation choice,
stated as such.

**Statistical note:** N_OUTER_FOLDS outer folds give a leak-free but
small-n performance distribution; paired significance tests across folds are
correspondingly limited in power (Module G reports effect sizes and bootstrap
intervals alongside p-values for this reason). Repeated nested CV would raise
that power at a proportional compute cost — a knob, not a default.

In [ ]:
def get_scaffold_groups(labels_df):
    '''Integer scaffold-group array for GroupKFold, derived from notebook
    02's precomputed global_scaffold_id column — never recomputed via RDKit
    here (compute-once-reuse-everywhere; avoids silent drift from the
    diversity reporting in notebook 02).'''
    codes, _ = pd.factorize(labels_df['global_scaffold_id'])
    return codes


def scaffold_train_test_split(labels_df, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    '''Scaffold-grouped train/test split using precomputed global_scaffold_id:
    whole scaffolds are RANDOMLY assigned to the test set (order shuffled by
    random_state) until test reaches test_size, guaranteeing no scaffold appears
    in both partitions. Reproducible for a given random_state — the canonical
    split fixes it; the repeated-benchmark repeats vary it (random_state=repeat).

    NOT class-stratified: scaffolds are the atomic unit, so test class balance
    can vary between splits — it is REPORTED per split (see split_composition)
    rather than enforced, so any instability (e.g. a nearly single-class test
    fold for a thin combination) is visible/auditable rather than hidden.'''
    scaffold_to_indices = defaultdict(list)
    for i, sid in enumerate(labels_df['global_scaffold_id'].values):
        scaffold_to_indices[sid].append(i)

    scaffolds = list(scaffold_to_indices.keys())
    rng = np.random.default_rng(random_state)
    rng.shuffle(scaffolds)

    n_total = len(labels_df)
    test_cutoff = int(n_total * test_size)
    train_idx, test_idx = [], []
    for sid in scaffolds:
        indices = scaffold_to_indices[sid]
        if len(test_idx) < test_cutoff:
            test_idx.extend(indices)
        else:
            train_idx.extend(indices)

    return np.array(sorted(train_idx)), np.array(sorted(test_idx))


def assert_no_scaffold_leakage(labels_df, idx_a, idx_b):
    '''Cheap set-intersection check on precomputed global_scaffold_id — no
    RDKit recomputation needed. Fails loud rather than letting a silent
    leakage bug propagate into a reported number.'''
    scaffolds_a = set(labels_df['global_scaffold_id'].values[idx_a])
    scaffolds_b = set(labels_df['global_scaffold_id'].values[idx_b])
    overlap = scaffolds_a & scaffolds_b
    assert not overlap, f'Scaffold leakage: {len(overlap)} scaffold(s) in both sets, e.g. {next(iter(overlap))!r}'


def canonical_splits(labels_df, cfg):
    '''The single canonical scaffold split used by every DEPLOYED / CALIBRATED
    / EXPLAINED model in this notebook. train/test first (scaffold-disjoint),
    then a calibration holdout carved OUT OF train only (never touching test).

    The deployed model is fit on cal_train_idx and calibrated on cal_holdout_idx,
    so the saved model file and its saved Platt/conformal calibrator are always
    the SAME underlying object trained on the SAME data — no mismatch between
    "the model" and "its calibrator" for downstream reuse (e.g. screening).

    Returns (train_idx, test_idx, cal_train_idx, cal_holdout_idx), all indices
    into labels_df. cal_train_idx | cal_holdout_idx partition train_idx; both
    are scaffold-disjoint from test_idx (since train_idx is).
    '''
    train_idx, test_idx = scaffold_train_test_split(labels_df, cfg['test_size'], cfg['random_state'])
    cal_tr_rel, cal_hold_rel = scaffold_train_test_split(
        labels_df.iloc[train_idx].reset_index(drop=True), test_size=0.2,
        random_state=cfg['random_state'] + 1000)
    cal_train_idx = train_idx[cal_tr_rel]
    cal_holdout_idx = train_idx[cal_hold_rel]
    return train_idx, test_idx, cal_train_idx, cal_holdout_idx


print('Scaffold-split helpers defined (reusing notebook 02 global_scaffold_id).')





In [ ]:
# =============================================================================
# OPTUNA OBJECTIVE FUNCTIONS — scaffold GroupKFold based, one per
# (algorithm, task). Generalized from the predecessor: no is_cb2 branching,
# identical search space and trial budget for every target/pool. class_weight
# / scale_pos_weight are tuned here AND are the only hyperparameter path that
# is used in optuna mode (and the 'fixed' mode's FIXED_HYPERPARAMS also carry
# class weighting / scale_pos_weight, so neither path silently drops class
# balancing — fixes lesson 2).
# =============================================================================

def objective_rf_classification(trial, X, y, groups, cv_folds, random_state):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 35),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5, 0.7]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample', None]),
        'random_state': random_state, 'n_jobs': 1,
    }
    model = RandomForestClassifier(**params)
    cv = GroupKFold(n_splits=cv_folds)
    try:
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring='roc_auc', n_jobs=OPTUNA_CV_N_JOBS, error_score=0.5)
        return float(scores.mean())
    except Exception:
        return 0.5


def objective_xgb_classification(trial, X, y, groups, cv_folds, random_state):
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    empirical_ratio = n_neg / n_pos if n_pos > 0 else 1.0
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.15, 0.5),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.3, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight',
            min(0.5, empirical_ratio * 0.5), max(2.0, empirical_ratio * 2.0), log=True),
        'random_state': random_state, 'n_jobs': 1, 'eval_metric': 'logloss',
    }
    model = xgb.XGBClassifier(**params)
    cv = GroupKFold(n_splits=cv_folds)
    try:
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring='roc_auc', n_jobs=OPTUNA_CV_N_JOBS, error_score=0.5)
        return float(scores.mean())
    except Exception:
        return 0.5


def objective_lgb_classification(trial, X, y, groups, cv_folds, random_state):
    max_depth = trial.suggest_int('max_depth', 5, 25)
    num_leaves = min(trial.suggest_int('num_leaves', 20, 150), 2 ** max_depth - 1)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'num_leaves': num_leaves, 'max_depth': max_depth,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.08, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0), 'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.15, 0.5),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', None]),
        'random_state': random_state, 'n_jobs': 1, 'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    cv = GroupKFold(n_splits=cv_folds)
    try:
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring='roc_auc', n_jobs=OPTUNA_CV_N_JOBS, error_score=0.5)
        return float(scores.mean())
    except Exception:
        return 0.5


def objective_rf_regression(trial, X, y, groups, cv_folds, random_state):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 35),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 0.3, 0.5, 0.7, 1.0]),
        'random_state': random_state, 'n_jobs': 1,
    }
    model = RandomForestRegressor(**params)
    cv = GroupKFold(n_splits=cv_folds)
    try:
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring='r2', n_jobs=OPTUNA_CV_N_JOBS, error_score=0.0)
        return float(scores.mean())
    except Exception:
        return 0.0


def objective_xgb_regression(trial, X, y, groups, cv_folds, random_state):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.15, 0.5),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.3, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'random_state': random_state, 'n_jobs': 1,
    }
    model = xgb.XGBRegressor(**params)
    cv = GroupKFold(n_splits=cv_folds)
    try:
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring='r2', n_jobs=OPTUNA_CV_N_JOBS, error_score=0.0)
        return float(scores.mean())
    except Exception:
        return 0.0


def objective_lgb_regression(trial, X, y, groups, cv_folds, random_state):
    max_depth = trial.suggest_int('max_depth', 5, 25)
    num_leaves = min(trial.suggest_int('num_leaves', 20, 150), 2 ** max_depth - 1)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'num_leaves': num_leaves, 'max_depth': max_depth,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.08, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0), 'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.15, 0.5),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'random_state': random_state, 'n_jobs': 1, 'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    cv = GroupKFold(n_splits=cv_folds)
    try:
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring='r2', n_jobs=OPTUNA_CV_N_JOBS, error_score=0.0)
        return float(scores.mean())
    except Exception:
        return 0.0


OBJECTIVE_FUNCTIONS = {
    ('rf', 'classification'): objective_rf_classification,
    ('xgb', 'classification'): objective_xgb_classification,
    ('lgb', 'classification'): objective_lgb_classification,
    ('rf', 'regression'): objective_rf_regression,
    ('xgb', 'regression'): objective_xgb_regression,
    ('lgb', 'regression'): objective_lgb_regression,
}
print(f'{len(OBJECTIVE_FUNCTIONS)} Optuna objective functions defined (3 algorithms x 2 tasks).')







In [ ]:
def tune_all_models(X, y_class, y_reg, groups_class, groups_reg, cfg, study_prefix):
    '''Run Optuna tuning for all 6 (algorithm, task) combinations, identical
    trial budget (N_OPTUNA_TRIALS) for every target/pool. Studies are persisted
    to SQLite and resumed with load_if_exists=True, so interrupted CHPC/PBS jobs
    continue from completed trials instead of restarting a study from scratch.'''
    sampler = TPESampler(seed=cfg['random_state'], constant_liar=True)
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)
    storage_path = cfg['results_dir'] / 'optuna_studies.sqlite3'
    # Lustre is a parallel filesystem; SQLite's default 5s busy-timeout
    # is too short for OPTUNA_PARALLEL_TRIALS concurrent writers to the
    # same study file, causing sporadic "database is locked" errors.
    # A longer busy-timeout makes SQLite retry instead of failing.
    storage = optuna.storages.RDBStorage(
        url=f'sqlite:///{storage_path}',
        engine_kwargs={'connect_args': {'timeout': 120}},
    )
    best_params = {}
    tuning_history = {}

    reg_mask = ~np.isnan(y_reg)
    X_reg, y_reg_valid, groups_reg_valid = X[reg_mask], y_reg[reg_mask], groups_reg[reg_mask]

    for (algo, task), objective_fn in OBJECTIVE_FUNCTIONS.items():
        if task == 'classification':
            X_t, y_t, g_t = X, y_class, groups_class
        else:
            X_t, y_t, g_t = X_reg, y_reg_valid, groups_reg_valid

        n_unique_groups = len(np.unique(g_t))
        cv_folds = min(cfg['optuna_cv_folds'], n_unique_groups)

        study_name = f'{study_prefix}_{algo}_{task}'
        study = optuna.create_study(
            study_name=study_name,
            direction='maximize', sampler=sampler, pruner=pruner,
            storage=storage, load_if_exists=True,
        )
        n_finished = sum(
            t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)
            for t in study.trials
        )
        remaining_trials = max(0, cfg['n_optuna_trials'] - n_finished)
        _t0 = time.time()
        if remaining_trials > 0:
            study.optimize(
                lambda trial, fn=objective_fn, X_=X_t, y_=y_t, g_=g_t, cvf=cv_folds: fn(
                    trial, X_, y_, g_, cvf, cfg['random_state']),
                n_trials=remaining_trials, n_jobs=OPTUNA_PARALLEL_TRIALS,
                show_progress_bar=False,
            )
        tune_seconds = time.time() - _t0

        # Persist the FULL per-trial history (objective value + every sampled
        # hyperparameter, every trial) — enables Optuna convergence diagnostics
        # (does the objective plateau before 50 trials?) and a real
        # across-target summary of the chosen hyperparameter VALUES, neither of
        # which is recoverable from best_params alone.
        trials_df = study.trials_dataframe()
        trials_path = cfg['results_dir'] / f'optuna_trials_{study_prefix}_{algo}_{task}.csv'
        trials_df.to_csv(trials_path, index=False)

        best_params[f'{algo}_{task}'] = study.best_params
        tuning_history[f'{algo}_{task}'] = {
            'best_value': study.best_value,
            'n_trials': len(study.trials),
            'best_params': study.best_params,
            'tune_seconds': tune_seconds,
            'trials_dataframe_path': str(trials_path),
            'optuna_storage': str(storage_path),
            'study_name': study_name,
            'n_finished_before_resume': int(n_finished),
            'n_trials_run_this_session': int(remaining_trials),
            'optuna_parallel_trials': int(OPTUNA_PARALLEL_TRIALS),
            'optuna_cv_n_jobs': int(OPTUNA_CV_N_JOBS),
        }
        print(f'  {ALGO_LABELS[algo]} ({task}): best CV score = {study.best_value:.4f} '
              f'over {len(study.trials)} total trials; ran {remaining_trials} this session '
              f'with {OPTUNA_PARALLEL_TRIALS} parallel trial(s) x {OPTUNA_CV_N_JOBS} CV worker(s) '
              f'({tune_seconds:.0f}s)')

    return best_params, tuning_history, {tag: OBJECTIVE_FUNCTIONS for tag in best_params}


def add_task_defaults(params, algo, task, random_state):
    '''Attach fixed (non-tuned) settings — n_jobs/verbosity/eval_metric —
    on top of Optuna's tuned params, without silently overwriting a tuned
    value. This is the ONLY place hyperparameters are assembled into a model
    call in this notebook: there is no separate 'default hyperparameters'
    path a model could train from instead.'''
    p = dict(params)
    p['random_state'] = random_state
    if algo == 'rf':
        p['n_jobs'] = -1
    elif algo == 'xgb':
        p['n_jobs'] = -1
        p.setdefault('eval_metric', 'logloss' if task == 'classification' else 'rmse')
    elif algo == 'lgb':
        p['n_jobs'] = -1
        p['verbose'] = -1
    return p

print('tune_all_models() and add_task_defaults() defined.')







In [ ]:
def build_model(algo, task, params):
    if algo == 'rf':
        return RandomForestClassifier(**params) if task == 'classification' else RandomForestRegressor(**params)
    if algo == 'xgb':
        return xgb.XGBClassifier(**params) if task == 'classification' else xgb.XGBRegressor(**params)
    if algo == 'lgb':
        return lgb.LGBMClassifier(**params) if task == 'classification' else lgb.LGBMRegressor(**params)
    raise ValueError(f'Unknown algorithm: {algo}')


CLASSIFICATION_METRICS = {
    'accuracy': lambda y, p: accuracy_score(y, (p >= 0.5).astype(int)),
    'roc_auc': lambda y, p: roc_auc_score(y, p),
    'pr_auc': lambda y, p: average_precision_score(y, p),
    'f1': lambda y, p: f1_score(y, (p >= 0.5).astype(int), zero_division=0),
    'precision': lambda y, p: precision_score(y, (p >= 0.5).astype(int), zero_division=0),
    'recall': lambda y, p: recall_score(y, (p >= 0.5).astype(int), zero_division=0),
    'brier': lambda y, p: brier_score_loss(y, p),
}
REGRESSION_METRICS = {
    'r2': lambda y, p: r2_score(y, p),
    'rmse': lambda y, p: float(np.sqrt(mean_squared_error(y, p))),
    'mae': lambda y, p: mean_absolute_error(y, p),
    'spearman_r': lambda y, p: float(spearmanr(y, p)[0]),
}


def safe_metric(fn, y, p):
    '''Evaluate a metric, returning NaN when it is mathematically undefined
    for this split (e.g. ROC-AUC when the outer-fold test set happens to
    contain only one class — an artifact of very small/imbalanced pools
    such as CCR5 ki, not a pipeline bug). NaN is preserved downstream so
    the gap is visible in every summary rather than silently dropped or
    replaced with a placeholder value.'''
    try:
        return fn(y, p)
    except ValueError:
        return np.nan


def evaluate_split(X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test,
                    hyperparams, random_state):
    '''Train all 6 (algorithm, task) models on one train/test split, return
    a dict of {(algo, task): {metric: value}} plus the fitted model objects.'''
    results = {}
    models = {}

    reg_train_mask = ~np.isnan(y_reg_train)
    reg_test_mask = ~np.isnan(y_reg_test)
    X_reg_train, y_reg_train_valid = X_train[reg_train_mask], y_reg_train[reg_train_mask]
    X_reg_test, y_reg_test_valid = X_test[reg_test_mask], y_reg_test[reg_test_mask]

    for algo in ALGORITHMS:
        clf_params = add_task_defaults(hyperparams[f'{algo}_classification'], algo, 'classification', random_state)
        clf = build_model(algo, 'classification', clf_params)
        clf.fit(X_train, y_class_train)
        proba = clf.predict_proba(X_test)[:, 1]
        results[(algo, 'classification')] = {m: safe_metric(fn, y_class_test, proba) for m, fn in CLASSIFICATION_METRICS.items()}
        models[(algo, 'classification')] = clf

        reg_params = add_task_defaults(hyperparams[f'{algo}_regression'], algo, 'regression', random_state)
        reg = build_model(algo, 'regression', reg_params)
        reg.fit(X_reg_train, y_reg_train_valid)
        pred = reg.predict(X_reg_test)
        results[(algo, 'regression')] = {m: safe_metric(fn, y_reg_test_valid, pred) for m, fn in REGRESSION_METRICS.items()}
        models[(algo, 'regression')] = reg

    return results, models

print('Model-building and single-split evaluation functions defined.')





In [ ]:
def tune_on_subset(X_combined, labels_df, subset_idx, cfg, study_prefix):
    '''Tune all 6 models on ONLY subset_idx (inner scaffold GroupKFold on the
    combined representation). subset_idx defines the data the tuner is allowed
    to see: an outer-train fold (nested CV) or the canonical cal_train partition
    (deployment). Nothing outside subset_idx — in particular no outer-test /
    canonical-test compound — ever enters hyperparameter selection, so the
    resulting hyperparameters carry no information about the data they are later
    evaluated on. Returns (best_params, tuning_history).'''
    sub_labels = labels_df.iloc[subset_idx].reset_index(drop=True)
    y_class = sub_labels['activity'].to_numpy(int)
    y_reg = sub_labels['pActivity'].to_numpy(float)
    groups_class = get_scaffold_groups(sub_labels)
    # Pass full-length scaffold groups. tune_all_models() applies the regression
    # non-NaN mask internally, so pre-filtering here would double-filter and
    # misalign groups whenever pActivity has missing values.
    groups_reg = groups_class

    if TUNING_MODE == 'fixed':
        # No hyperparameter search. Use FIXED_HYPERPARAMS for every model, and
        # score each once via the same inner scaffold GroupKFold used for
        # tuning — that inner-CV score (leak-free, subset training data only) is
        # what best-algorithm selection reads downstream, so selection still
        # works identically without any tuning. XGBoost's scale_pos_weight is
        # set here to the subset's empirical neg/pos ratio (a data-derived
        # imbalance setting, not a tuned hyperparameter) so class imbalance is
        # handled rather than silently dropped.
        X_sub = X_combined[subset_idx]
        n_pos = int((y_class == 1).sum()); n_neg = int((y_class == 0).sum())
        empirical_ratio = (n_neg / n_pos) if n_pos > 0 else 1.0
        best_params = {}
        tuning_history = {}
        reg_mask = ~np.isnan(y_reg)
        for (algo, task) in OBJECTIVE_FUNCTIONS:
            params = dict(FIXED_HYPERPARAMS[f'{algo}_{task}'])
            if algo == 'xgb' and task == 'classification':
                params['scale_pos_weight'] = float(empirical_ratio)
            best_params[f'{algo}_{task}'] = params

            if task == 'classification':
                X_t, y_t, g_t = X_sub, y_class, groups_class
            else:
                X_t, y_t, g_t = X_sub[reg_mask], y_reg[reg_mask], groups_class[reg_mask]
            n_groups = len(np.unique(g_t))
            cv_folds = min(cfg['optuna_cv_folds'], n_groups)
            score = np.nan
            if cv_folds >= 2:
                model = build_model(algo, task, add_task_defaults(params, algo, task, cfg['random_state']))
                cv = GroupKFold(n_splits=cv_folds)
                scoring = 'roc_auc' if task == 'classification' else 'r2'
                default_score = 0.5 if task == 'classification' else 0.0
                try:
                    scores = cross_val_score(model, X_t, y_t, cv=cv, groups=g_t,
                                             scoring=scoring, n_jobs=N_CORES, error_score=default_score)
                    score = float(scores.mean())
                except Exception:
                    score = default_score
            tuning_history[f'{algo}_{task}'] = {
                'best_value': score, 'n_trials': 0, 'best_params': params,
                'tune_seconds': 0.0, 'trials_dataframe_path': None, 'mode': 'fixed',
            }
        return best_params, tuning_history

    best_params, tuning_history, _ = tune_all_models(
        X_combined[subset_idx], y_class, y_reg, groups_class, groups_reg, cfg, study_prefix)
    return best_params, tuning_history


def run_nested_cv(target, pool, cfg):
    '''Leak-free nested scaffold cross-validation for one (target, pool).

    Outer loop: GroupKFold over scaffold groups (whole scaffolds held out per
    fold). For each outer fold: tune hyperparameters on the OUTER-TRAIN fold
    only (inner scaffold GroupKFold, N_OPTUNA_TRIALS on the combined
    representation), then train every algorithm x representation on outer-train
    and evaluate ONCE on the untouched outer-test. No outer-test compound ever
    influences tuning, so the outer-test metric distribution is an unbiased
    (leak-free) generalization estimate, and algorithm comparison is done on
    outer-test WITHOUT any prior algorithm selection on that same test.

    Returns (metrics_df, inner_cv_df, importance_records, test_composition_df, prediction_df).
    '''
    labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')
    y_class_all = labels_df['activity'].to_numpy(int)
    y_reg_all = labels_df['pActivity'].to_numpy(float)
    groups_all = get_scaffold_groups(labels_df)
    X_by_rep = {rep: np.load(cfg['features_dir'] / f'{rep}.npy') for rep in FEATURE_REPS}

    n_scaffolds = len(np.unique(groups_all))
    n_outer = min(N_OUTER_FOLDS, n_scaffolds)
    outer = GroupKFold(n_splits=n_outer)

    metric_records, inner_cv_records, importance_records, comp_records, pred_records = [], [], [], [], []
    for fold_i, (train_idx, test_idx) in enumerate(
            outer.split(labels_df, y_class_all, groups_all)):
        assert_no_scaffold_leakage(labels_df, train_idx, test_idx)

        fold_params, fold_hist = tune_on_subset(
            X_by_rep['combined'], labels_df, train_idx, cfg,
            study_prefix=f'{cfg["combo_key"]}_outer{fold_i}')

        for model_task, info in fold_hist.items():
            inner_cv_records.append({
                'target': target, 'activity_pool': pool, 'outer_fold': fold_i,
                'model_task': model_task, 'inner_cv_best_value': info['best_value']})

        yte = y_class_all[test_idx]
        comp_records.append({
            'target': target, 'activity_pool': pool, 'outer_fold': fold_i,
            'n_train': len(train_idx), 'n_test': len(test_idx),
            'n_test_active': int(yte.sum()), 'n_test_inactive': int((yte == 0).sum()),
            'pct_test_active': float(yte.mean() * 100) if len(yte) else np.nan})

        for rep in FEATURE_REPS:
            X = X_by_rep[rep]
            results, models = evaluate_split(
                X[train_idx], X[test_idx],
                y_class_all[train_idx], y_class_all[test_idx],
                y_reg_all[train_idx], y_reg_all[test_idx],
                fold_params, cfg['random_state'])
            for (algo, task), metrics in results.items():
                for metric_name, metric_value in metrics.items():
                    metric_records.append({
                        'target': target, 'activity_pool': pool,
                        'feature_representation': rep, 'algorithm': ALGO_LABELS[algo], 'task': task,
                        'outer_fold': fold_i, 'n_train': len(train_idx), 'n_test': len(test_idx),
                        'metric_name': metric_name, 'metric_value': metric_value})
                model = models[(algo, task)]
                if hasattr(model, 'feature_importances_'):
                    importance_records.append({
                        'feature_representation': rep, 'algorithm': ALGO_LABELS[algo], 'task': task,
                        'outer_fold': fold_i, 'importances': model.feature_importances_.tolist()})
                if task == 'classification':
                    proba = model.predict_proba(X[test_idx])[:, 1]
                    pred = (proba >= 0.5).astype(int)
                    for row_i, test_i in enumerate(test_idx):
                        pred_records.append({
                            'target': target, 'activity_pool': pool,
                            'feature_representation': rep, 'algorithm': ALGO_LABELS[algo],
                            'outer_fold': fold_i,
                            'global_compound_id': labels_df['global_compound_id'].iloc[test_i],
                            'molecule_chembl_id': labels_df['molecule_chembl_id'].iloc[test_i],
                            'y_true_class': int(y_class_all[test_i]),
                            'predicted_proba': float(proba[row_i]),
                            'predicted_class': int(pred[row_i]),
                            'is_misclassified': bool(pred[row_i] != y_class_all[test_i]),
                        })

    return (pd.DataFrame(metric_records), pd.DataFrame(inner_cv_records),
            importance_records, pd.DataFrame(comp_records), pd.DataFrame(pred_records))

print('tune_on_subset() and run_nested_cv() defined.')






In [ ]:
# =============================================================================
# MASTER BENCHMARK LOOP — leak-free by construction. Two tuning contexts per
# (target, pool), both tuning on TRAINING data only (never their own test):
#   (1) Nested CV (N_OUTER_FOLDS outer scaffold folds, inner tuning per fold)
#       -> the unbiased cross-target performance distribution + algorithm
#          comparison. This is the headline generalization estimate.
#   (2) Deployment tuning on the canonical cal_train partition only -> the
#       hyperparameters for the single deployed/calibrated/explained model in
#       Modules E/F/H, whose canonical test set the tuning never saw.
# Both contexts are checkpointed per combination so a restart resumes.
# =============================================================================
benchmark_long_frames = []
inner_cv_frames = []
split_composition_frames = []
nested_prediction_frames = []
all_hyperparams = {}   # DEPLOYMENT hyperparameters (tuned on canonical cal_train)
all_inner_cv = {}      # deployment inner-CV best_value per model_task (leak-free
                       # signal for algorithm selection — never uses test data)

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        cfg['models_dir'].mkdir(parents=True, exist_ok=True)
        cfg['results_dir'].mkdir(parents=True, exist_ok=True)
        combo_key = cfg['combo_key']
        print(f'\n{"="*70}\nBENCHMARKING (nested CV): {target} / {pool}\n{"="*70}')

        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')

        # ---- (1) Nested CV: leak-free performance estimate ----
        nested_ckpt = cfg['results_dir'] / 'nested_cv_metrics.csv'
        nested_inner_ckpt = cfg['results_dir'] / 'nested_cv_inner_scores.csv'
        nested_comp_ckpt = cfg['results_dir'] / 'nested_cv_test_composition.csv'
        nested_pred_ckpt = cfg['results_dir'] / 'nested_cv_predictions.csv'
        nested_complete = all(p.exists() for p in [nested_ckpt, nested_inner_ckpt, nested_comp_ckpt, nested_pred_ckpt])
        if nested_complete:
            print('  Nested-CV checkpoint found, loading cached results')
            nested_df = pd.read_csv(nested_ckpt)
            inner_cv_df = pd.read_csv(nested_inner_ckpt)
            comp_df = pd.read_csv(nested_comp_ckpt)
            pred_df = pd.read_csv(nested_pred_ckpt)
        else:
            print(f'  Running {N_OUTER_FOLDS}-fold nested CV (tuning inside each outer fold)...')
            nested_df, inner_cv_df, importance_records, comp_df, pred_df = run_nested_cv(target, pool, cfg)
            nested_df.to_csv(nested_ckpt, index=False)
            inner_cv_df.to_csv(nested_inner_ckpt, index=False)
            comp_df.to_csv(nested_comp_ckpt, index=False)
            pred_df.to_csv(nested_pred_ckpt, index=False)
            for rep in FEATURE_REPS:
                recs = [r for r in importance_records if r['feature_representation'] == rep]
                with open(cfg['results_dir'] / f'nested_importances_{rep}.json', 'w') as f:
                    json.dump(recs, f)
        benchmark_long_frames.append(nested_df)
        inner_cv_frames.append(inner_cv_df)
        split_composition_frames.append(comp_df)
        nested_prediction_frames.append(pred_df)
        log_milestone(target, pool, 'Nested-CV benchmarking complete')

        # ---- (2) Deployment tuning: canonical cal_train ONLY (leak-free wrt
        # the canonical test used by Modules E/F/H) ----
        _, _, cal_train_idx, _ = canonical_splits(labels_df, cfg)
        dep_params_path = cfg['models_dir'] / 'optuna_best_params.json'
        dep_hist_path = cfg['results_dir'] / 'optuna_tuning_history.json'
        if dep_params_path.exists():
            with open(dep_params_path) as f: hyperparams = json.load(f)
            with open(dep_hist_path) as f: dep_hist = json.load(f)
        else:
            X_combined = np.load(cfg['features_dir'] / 'combined.npy')
            print('  Deployment tuning on canonical cal_train (combined features)...')
            hyperparams, dep_hist = tune_on_subset(
                X_combined, labels_df, cal_train_idx, cfg, study_prefix=f'{combo_key}_deploy')
            with open(dep_params_path, 'w') as f: json.dump(hyperparams, f, indent=2)
            with open(dep_hist_path, 'w') as f: json.dump(dep_hist, f, indent=2, default=str)
        all_hyperparams[combo_key] = hyperparams
        all_inner_cv[combo_key] = {mt: dep_hist[mt]['best_value'] for mt in dep_hist}
        log_milestone(target, pool, 'Deployment hyperparameter tuning complete')

benchmark_results_long = pd.concat(benchmark_long_frames, ignore_index=True)
benchmark_long_path = results_path / shard_name('benchmark_results_long.csv')
benchmark_results_long.to_csv(benchmark_long_path, index=False)
_register('benchmark_results_long.csv', benchmark_long_path, len(benchmark_results_long))

inner_cv_all_df = pd.concat(inner_cv_frames, ignore_index=True)
inner_cv_all_path = results_path / shard_name('nested_cv_inner_scores_all.csv')
inner_cv_all_df.to_csv(inner_cv_all_path, index=False)
_register('nested_cv_inner_scores_all.csv', inner_cv_all_path, len(inner_cv_all_df))

split_composition_df = pd.concat(split_composition_frames, ignore_index=True)
split_composition_path = results_path / shard_name('split_composition.csv')
split_composition_df.to_csv(split_composition_path, index=False)
_register('split_composition.csv', split_composition_path, len(split_composition_df))

nested_predictions_df = pd.concat(nested_prediction_frames, ignore_index=True)
nested_predictions_path = results_path / shard_name('nested_cv_predictions_all.csv')
nested_predictions_df.to_csv(nested_predictions_path, index=False)
_register('nested_cv_predictions_all.csv', nested_predictions_path, len(nested_predictions_df))

print(f'\nLeak-free nested-CV benchmark table saved: {benchmark_long_path} ({len(benchmark_results_long)} rows)')
print(f'Per-outer-fold test class balance saved: {split_composition_path} ({len(split_composition_df)} rows)')
print(f'Leak-free nested-CV per-compound predictions saved: {nested_predictions_path} ({len(nested_predictions_df)} rows)')
print(f'Columns: {list(benchmark_results_long.columns)}')







In [ ]:
# ---- Best algorithm per (target, pool, task), selected by DEPLOYMENT
# inner-CV score (Optuna best_value on cal_train) — never from test
# performance. This fixes the selection-on-test leak: the algorithm used for
# every single-model downstream analysis (SHAP, Y-randomization, deployed
# model, error analysis) is chosen using only training-internal cross-
# validation. Selection is on the combined representation (matching the
# deployment tuning) and applied to each representation's deployed model. ----
best_algorithm = {}
best_algo_rows = []
for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        inner = all_inner_cv.get(cfg['combo_key'], {})
        for task in TASKS:
            scored = [(a, inner.get(f'{a}_{task}', float('-inf'))) for a in ALGORITHMS]
            best_algo_key = max(scored, key=lambda kv: kv[1])[0]
            for rep in FEATURE_REPS:
                best_algorithm[(target, pool, rep, task)] = ALGO_LABELS[best_algo_key]
            best_algo_rows.append({
                'target': target, 'activity_pool': pool, 'task': task,
                'best_algorithm': ALGO_LABELS[best_algo_key],
                'selected_by': 'deployment inner-CV best_value (leak-free)',
            })

best_algorithm_df = pd.DataFrame(best_algo_rows)
best_algo_path = results_path / shard_name('best_algorithm_by_combination.csv')
best_algorithm_df.to_csv(best_algo_path, index=False)
_register('best_algorithm_by_combination.csv', best_algo_path, len(best_algorithm_df))
print(best_algorithm_df.to_string(index=False))





In [ ]:
# ---- Figure: cross-target consistency, combined representation, full
# pool -- the core Q1 figure (does the same workflow perform consistently
# across GPCRs). Plot-source data is benchmark_results_long.csv itself. ----
plot_df = benchmark_results_long[
    (benchmark_results_long['feature_representation'] == 'combined') &
    (benchmark_results_long['activity_pool'] == 'full') &
    (benchmark_results_long['task'] == 'classification') &
    (benchmark_results_long['metric_name'] == 'roc_auc')
]

fig, ax = plt.subplots(figsize=(8, 5))
positions = []
box_data = []
box_labels = []
pos = 0
for target in TARGETS:
    for algo_label in [ALGO_LABELS[a] for a in ALGORITHMS]:
        sub = plot_df[(plot_df['target'] == target) & (plot_df['algorithm'] == algo_label)]
        if len(sub) == 0:
            continue
        box_data.append(sub['metric_value'].values)
        box_labels.append(f'{target.upper()}\n{algo_label}')
        positions.append(pos)
        pos += 1
    pos += 1  # gap between targets

bp = ax.boxplot(box_data, positions=positions, widths=0.6, patch_artist=True, showfliers=False)
for i, (patch, target) in enumerate([(p, TARGETS[i // 3]) for i, p in enumerate(bp['boxes'])]):
    patch.set_facecolor(TARGET_COLORS[target])
    patch.set_alpha(0.7)
ax.set_xticks(positions)
ax.set_xticklabels(box_labels, fontsize=7, rotation=90)
ax.set_ylabel('Scaffold-split ROC-AUC (nested-CV outer folds)')
panel_label(ax, 'A')
fig.tight_layout()
save_fig(fig, 'figure_cross_target_classification_performance', fig_dir)
_register('figure_cross_target_classification_performance.png', fig_dir / 'figure_cross_target_classification_performance.png')
_register('figure_cross_target_classification_performance.pdf', fig_dir / 'figure_cross_target_classification_performance.pdf')
plot_df.to_csv(fig_dir / shard_name('figure_cross_target_classification_performance_data.csv'), index=False)
_register('figure_cross_target_classification_performance_data.csv', fig_dir / shard_name('figure_cross_target_classification_performance_data.csv'), len(plot_df))
print('Figure saved: figure_cross_target_classification_performance (+ plot-source CSV)')





In [ ]:
# ---- Figure: activity-pool sensitivity (Ki vs Ki+IC50 vs Ki+IC50+EC50),
# combined representation, mean best-algorithm ROC-AUC per target -- the
# three-way activity-type-harmonisation sensitivity analysis notebook 02
# set up the pools for. Uses POOL_LABELS (never raw 'ki'/'ki_ic50'/'full'
# codes) on the axis.
#
# Disclosed exception (not a bug): CCR5's 'ki'-only pool
# (154 compounds) is too thin for reliable scaffold-split modeling and is
# excluded from THIS sensitivity comparison specifically -- it is still
# trained and reported everywhere else in this notebook (Modules D-H),
# just not used to judge activity-pool sensitivity, where its small-sample
# noise would be misread as a real pool effect. ----
POOL_SENSITIVITY_EXCLUDE = {('ccr5', 'ki')}

pool_sensitivity_rows = []
for target, pool in RUN_COMBOS:
        if (target, pool) in POOL_SENSITIVITY_EXCLUDE:
            print(f'  Excluding {target}/{pool} from activity-pool sensitivity comparison '
                  f'(disclosed exception: too few compounds for reliable scaffold-split comparison)')
            continue
        best_label = best_algorithm.get((target, pool, 'combined', 'classification'))
        if best_label is None:
            continue
        sub = benchmark_results_long[
            (benchmark_results_long['target'] == target) & (benchmark_results_long['activity_pool'] == pool) &
            (benchmark_results_long['feature_representation'] == 'combined') &
            (benchmark_results_long['task'] == 'classification') & (benchmark_results_long['metric_name'] == 'roc_auc') &
            (benchmark_results_long['algorithm'] == best_label)
        ]
        if len(sub) == 0:
            continue
        pool_sensitivity_rows.append({
            'target': target, 'activity_pool': pool, 'best_algorithm': best_label,
            'mean_roc_auc': float(sub['metric_value'].mean()), 'std_roc_auc': float(sub['metric_value'].std()),
        })
pool_sensitivity_df = pd.DataFrame(pool_sensitivity_rows)

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(POOLS))
width = 0.15
for i, target in enumerate(TARGETS):
    sub = pool_sensitivity_df[pool_sensitivity_df['target'] == target].set_index('activity_pool').reindex(POOLS)
    ax.bar(x + (i - 2) * width, sub['mean_roc_auc'], width, yerr=sub['std_roc_auc'],
           label=target.upper(), color=TARGET_COLORS[target], alpha=0.85, capsize=2)
ax.set_xticks(x)
ax.set_xticklabels([POOL_LABELS[p] for p in POOLS])
ax.set_ylabel('Mean scaffold-split ROC-AUC (best algorithm, nested-CV outer folds)')
ax.legend(frameon=False, fontsize=8, ncol=2)
panel_label(ax, 'A')
fig.tight_layout()
save_fig(fig, 'figure_activity_pool_sensitivity', fig_dir)
_register('figure_activity_pool_sensitivity.png', fig_dir / 'figure_activity_pool_sensitivity.png')
_register('figure_activity_pool_sensitivity.pdf', fig_dir / 'figure_activity_pool_sensitivity.pdf')
pool_sensitivity_path = results_path / shard_name('activity_pool_sensitivity_summary.csv')
pool_sensitivity_df.to_csv(pool_sensitivity_path, index=False)
_register('activity_pool_sensitivity_summary.csv', pool_sensitivity_path, len(pool_sensitivity_df))
print('Figure saved: figure_activity_pool_sensitivity (+ activity_pool_sensitivity_summary.csv)')





## Module E: Model Reliability

Calibration (reliability curves, ECE, MCE, Brier, calibration slope/
intercept — via **Venn-ABERS**, `venn_abers.VennAbers`, fit on the
calibration holdout and applied to test), conformal prediction (`crepes` —
coverage, efficiency, stratified by activity level and scaffold-singleton
status), and applicability domain (kNN-distance based).

**Switched from Platt/sigmoid scaling to Venn-ABERS (2026-08)**, after
live research surfaced the closest published analog to this exact task
(AstraZeneca/JCIM, ~40M compound-target pairs, both stratified AND
scaffold-split validated) showing Venn-ABERS roughly halving Brier score
vs Platt scaling for bioactivity classification specifically. Venn-ABERS
is fit directly on the 2-column `predict_proba` output (no logit
transform) and returns a `[p0, p1]` validity interval per compound in
addition to the point-estimate probability `p_prime` — that interval
width is a real per-compound calibration-uncertainty signal Platt scaling
has no equivalent for, persisted here (`venn_abers_intervals.csv`) rather
than discarded. `crepes`' conformal-prediction wrappers are unaffected by
this swap (a separate calibration mechanism, no native Venn-ABERS support
in `crepes` — confirmed by direct inspection of `crepes.extras`).

**Applicability domain is computed exactly ONCE per (target, pool,
representation)** and frozen as a single joblib artifact
(`ml/models/{target}_{pool}/{representation}/ad_reference.pkl`) that every
downstream stage (Module H, the later DrugBank notebook) loads read-only —
this fixes an earlier implementation issue, where `run_publication_analysis`
fit its own scaler+kNN internally and `handle_drugbank_screening` then
refit an entirely separate scaler+kNN from scratch with a comment
literally acknowledging the duplication.


In [ ]:
def expected_calibration_error(y_true, y_proba, n_bins=10):
    '''ECE: bin predictions, weight each bin's |accuracy - confidence| by
    the bin's share of samples.'''
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(y_proba, bin_edges) - 1, 0, n_bins - 1)
    ece, mce = 0.0, 0.0
    n = len(y_true)
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.sum() == 0:
            continue
        bin_acc = y_true[mask].mean()
        bin_conf = y_proba[mask].mean()
        gap = abs(bin_acc - bin_conf)
        ece += (mask.sum() / n) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)


def calibration_slope_intercept(y_true, y_proba):
    '''Logistic recalibration slope/intercept: fit y ~ logit(p) via a
    1-feature logistic regression. Slope=1, intercept=0 is perfect
    calibration; slope<1 indicates overconfidence.'''
    from sklearn.linear_model import LogisticRegression
    eps = 1e-6
    p_clipped = np.clip(y_proba, eps, 1 - eps)
    logit_p = np.log(p_clipped / (1 - p_clipped)).reshape(-1, 1)
    try:
        lr = LogisticRegression()
        lr.fit(logit_p, y_true)
        return float(lr.coef_[0][0]), float(lr.intercept_[0])
    except Exception:
        return float('nan'), float('nan')


def calibration_report(y_true, y_proba_raw, y_proba_calibrated, n_bins=10):
    ece_raw, mce_raw = expected_calibration_error(y_true, y_proba_raw, n_bins)
    ece_cal, mce_cal = expected_calibration_error(y_true, y_proba_calibrated, n_bins)
    slope_raw, intercept_raw = calibration_slope_intercept(y_true, y_proba_raw)
    slope_cal, intercept_cal = calibration_slope_intercept(y_true, y_proba_calibrated)
    return {
        'brier_raw': float(brier_score_loss(y_true, y_proba_raw)),
        'brier_calibrated': float(brier_score_loss(y_true, y_proba_calibrated)),
        'ece_raw': ece_raw, 'mce_raw': mce_raw,
        'ece_calibrated': ece_cal, 'mce_calibrated': mce_cal,
        'calibration_slope_raw': slope_raw, 'calibration_intercept_raw': intercept_raw,
        'calibration_slope_calibrated': slope_cal, 'calibration_intercept_calibrated': intercept_cal,
    }

print('Calibration metric functions defined.')





In [ ]:
def fit_applicability_domain(X_train, k=5):
    '''kNN-distance-based applicability domain, fit on TRAINING data only.
    Threshold = mean + 2*SD of training compounds own mean-kNN-distance to
    their k nearest OTHER training neighbours (leave-self-out).'''
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train.astype(float))
    k_eff = min(k, len(X_scaled) - 2)
    knn = NearestNeighbors(n_neighbors=k_eff + 1, metric='euclidean', n_jobs=-1)
    knn.fit(X_scaled)
    train_dists, _ = knn.kneighbors(X_scaled)
    train_knn_mean = train_dists[:, 1:].mean(axis=1)  # exclude self (distance 0)
    threshold = float(np.mean(train_knn_mean) + 2 * np.std(train_knn_mean))
    return {
        'scaler': scaler, 'knn': knn, 'k': k_eff, 'threshold': threshold,
        'train_knn_mean_mean': float(np.mean(train_knn_mean)),
        'train_knn_mean_std': float(np.std(train_knn_mean)),
    }


def apply_applicability_domain(ad_ref, X_query):
    X_scaled = ad_ref['scaler'].transform(X_query.astype(float))
    dists, _ = ad_ref['knn'].kneighbors(X_scaled)
    knn_mean = dists[:, :ad_ref['k']].mean(axis=1)
    within = knn_mean <= ad_ref['threshold']
    return knn_mean, within

print('Applicability-domain fit/apply functions defined (fit-once, frozen artifact).')





In [ ]:
# =============================================================================
# MODULE E MASTER LOOP — for each (target, pool, representation): use the
# canonical scaffold split (repeat=0, the same split Module H's final models
# use) to fit calibration, conformal prediction, and AD ONCE, then persist.
# =============================================================================
calibration_records = []
conformal_records = []
reliability_curve_records = []  # per-bin accuracy-vs-confidence, so the reliability
                                # diagram is replottable without retraining
venn_abers_interval_records = []  # per-compound [p0,p1] validity intervals from
                                   # Venn-ABERS calibration, not just the summary stats

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')
        y_class = labels_df['activity'].to_numpy(int)
        y_reg = labels_df['pActivity'].to_numpy(float)

        # Canonical split: model fit on cal_train_idx, calibrated on
        # cal_holdout_idx, evaluated on test_idx (never touched for fitting or
        # calibration). Module H's deployed model uses the SAME cal_train_idx,
        # so the saved model matches this calibrator/conformal wrapper exactly.
        train_idx, test_idx, cal_train_idx, cal_holdout_idx = canonical_splits(labels_df, cfg)

        hyperparams = all_hyperparams[cfg['combo_key']]

        for representation in FEATURE_REPS:
            rep_dir = cfg['models_dir'] / representation
            rep_dir.mkdir(parents=True, exist_ok=True)
            results_rep_dir = cfg['results_dir'] / representation
            results_rep_dir.mkdir(parents=True, exist_ok=True)

            X = np.load(cfg['features_dir'] / f'{representation}.npy')

            # ---- Applicability domain: fit ONCE on the canonical training
            # partition, freeze as a single artifact. ----
            ad_ref = fit_applicability_domain(X[train_idx])
            joblib.dump(ad_ref, rep_dir / 'ad_reference.pkl')
            test_knn_dist, test_within_ad = apply_applicability_domain(ad_ref, X[test_idx])

            # ---- Calibration + conformal prediction, per algorithm ----
            for algo in ALGORITHMS:
                clf_params = add_task_defaults(hyperparams[f'{algo}_classification'], algo, 'classification', cfg['random_state'])
                clf = build_model(algo, 'classification', clf_params)
                clf.fit(X[cal_train_idx], y_class[cal_train_idx])

                proba_raw_holdout_2col = clf.predict_proba(X[cal_holdout_idx])
                proba_raw_test_2col = clf.predict_proba(X[test_idx])
                proba_raw_holdout = proba_raw_holdout_2col[:, 1]
                proba_raw_test = proba_raw_test_2col[:, 1]

                # Venn-ABERS calibration (replaces Platt/sigmoid scaling).
                # Fit on the calibration holdout's raw 2-column predict_proba
                # output directly -- no logit transform needed, unlike Platt.
                # Switched after live research found the closest published
                # analog to this exact task (AstraZeneca/JCIM, 40M
                # compound-target pairs, stratified AND scaffold-split
                # validated) showed Venn-ABERS roughly halving Brier score
                # vs Platt scaling for bioactivity classification. p0_p1 is
                # Venn-ABERS' own validity-interval width -- a real
                # per-compound calibration-uncertainty signal Platt scaling
                # has no equivalent for, persisted below rather than discarded.
                venn_abers_calibrator = VennAbers()
                venn_abers_calibrator.fit(proba_raw_holdout_2col, y_class[cal_holdout_idx])
                joblib.dump(venn_abers_calibrator, rep_dir / f'{algo}_clf_calibrator.pkl')

                p_prime_test, p0_p1_test = venn_abers_calibrator.predict_proba(proba_raw_test_2col)
                proba_cal_test = p_prime_test[:, 1]
                interval_width_test = p0_p1_test[:, 1] - p0_p1_test[:, 0]

                report = calibration_report(y_class[test_idx], proba_raw_test, proba_cal_test)
                report.update({'target': target, 'activity_pool': pool,
                               'feature_representation': representation, 'algorithm': ALGO_LABELS[algo],
                               'venn_abers_mean_interval_width': float(interval_width_test.mean()),
                               'venn_abers_std_interval_width': float(interval_width_test.std())})
                calibration_records.append(report)

                # Per-compound Venn-ABERS validity intervals -- persisted so
                # this calibration-uncertainty signal is available for
                # downstream analysis (e.g. per-compound reliability), not
                # discarded after only the summary mean/std above.
                test_global_ids = labels_df['global_compound_id'].to_numpy()[test_idx]
                for row_i in range(len(test_idx)):
                    venn_abers_interval_records.append({
                        'target': target, 'activity_pool': pool,
                        'feature_representation': representation, 'algorithm': ALGO_LABELS[algo],
                        'global_compound_id': test_global_ids[row_i],
                        'p0': float(p0_p1_test[row_i, 0]), 'p1': float(p0_p1_test[row_i, 1]),
                        'interval_width': float(interval_width_test[row_i]),
                        'calibrated_proba': float(proba_cal_test[row_i]),
                    })

                # Reliability-curve bins (accuracy vs predicted confidence),
                # raw and calibrated — persisted so the reliability diagram is
                # replottable later without retraining (a CB2 figure was lost
                # because only the rendered image, not these bins, was kept).
                for curve_kind, proba_curve in [('raw', proba_raw_test), ('calibrated', proba_cal_test)]:
                    try:
                        frac_pos, mean_pred = calibration_curve(
                            y_class[test_idx], proba_curve, n_bins=10, strategy='uniform')
                    except Exception:
                        continue
                    for mp, fp in zip(mean_pred, frac_pos):
                        reliability_curve_records.append({
                            'target': target, 'activity_pool': pool,
                            'feature_representation': representation, 'algorithm': ALGO_LABELS[algo],
                            'curve': curve_kind, 'mean_predicted_confidence': float(mp),
                            'observed_frequency': float(fp),
                        })

                # ---- Conformal prediction (crepes) — classification ----
                wrapped_clf = WrapClassifier(clf)
                wrapped_clf.calibrate(X[cal_holdout_idx], y_class[cal_holdout_idx])
                joblib.dump(wrapped_clf, rep_dir / f'{algo}_clf_conformal.pkl')
                for conf_level in (0.8, 0.9, 0.95):
                    # labels=False: return the binary (n_samples, n_classes) indicator
                    # array indexed by wrapped_clf class order, not the default labels=True
                    # list-of-label-lists format.
                    pred_sets = wrapped_clf.predict_set(X[test_idx], confidence=conf_level, labels=False)
                    covered = pred_sets[np.arange(len(test_idx)), y_class[test_idx]].astype(bool)
                    set_sizes = pred_sets.sum(axis=1)
                    for stratum_name, stratum_mask in [
                        ('overall', np.ones(len(test_idx), dtype=bool)),
                        ('active', y_class[test_idx] == 1),
                        ('inactive', y_class[test_idx] == 0),
                        ('within_ad', test_within_ad),
                        ('outside_ad', ~test_within_ad),
                        ('singleton_scaffold', labels_df['is_singleton_scaffold'].to_numpy()[test_idx]),
                        ('non_singleton_scaffold', ~labels_df['is_singleton_scaffold'].to_numpy()[test_idx]),
                    ]:
                        if stratum_mask.sum() == 0:
                            continue
                        conformal_records.append({
                            'target': target, 'activity_pool': pool,
                            'feature_representation': representation, 'algorithm': ALGO_LABELS[algo],
                            'task': 'classification', 'confidence_level': conf_level,
                            'stratum': stratum_name, 'n': int(stratum_mask.sum()),
                            'empirical_coverage': float(covered[stratum_mask].mean()),
                            'mean_set_size': float(set_sizes[stratum_mask].mean()),
                        })

                # ---- Conformal prediction (crepes) — regression ----
                reg_cal_mask = ~np.isnan(y_reg[cal_train_idx])
                reg_hold_mask = ~np.isnan(y_reg[cal_holdout_idx])
                reg_test_mask = ~np.isnan(y_reg[test_idx])
                reg_params = add_task_defaults(hyperparams[f'{algo}_regression'], algo, 'regression', cfg['random_state'])
                reg = build_model(algo, 'regression', reg_params)
                reg.fit(X[cal_train_idx][reg_cal_mask], y_reg[cal_train_idx][reg_cal_mask])
                wrapped_reg = WrapRegressor(reg)
                wrapped_reg.calibrate(X[cal_holdout_idx][reg_hold_mask], y_reg[cal_holdout_idx][reg_hold_mask])
                joblib.dump(wrapped_reg, rep_dir / f'{algo}_reg_conformal.pkl')
                X_test_reg = X[test_idx][reg_test_mask]
                y_test_reg = y_reg[test_idx][reg_test_mask]
                test_within_ad_reg = test_within_ad[reg_test_mask]
                singleton_test_reg = labels_df['is_singleton_scaffold'].to_numpy()[test_idx][reg_test_mask]
                for conf_level in (0.8, 0.9, 0.95):
                    intervals = wrapped_reg.predict_int(X_test_reg, confidence=conf_level)
                    lo, hi = intervals[:, 0], intervals[:, 1]
                    covered = (y_test_reg >= lo) & (y_test_reg <= hi)
                    widths = hi - lo
                    for stratum_name, stratum_mask in [
                        ('overall', np.ones(len(y_test_reg), dtype=bool)),
                        ('within_ad', test_within_ad_reg),
                        ('outside_ad', ~test_within_ad_reg),
                        ('singleton_scaffold', singleton_test_reg),
                        ('non_singleton_scaffold', ~singleton_test_reg),
                    ]:
                        if stratum_mask.sum() == 0:
                            continue
                        conformal_records.append({
                            'target': target, 'activity_pool': pool,
                            'feature_representation': representation, 'algorithm': ALGO_LABELS[algo],
                            'task': 'regression', 'confidence_level': conf_level,
                            'stratum': stratum_name, 'n': int(stratum_mask.sum()),
                            'empirical_coverage': float(covered[stratum_mask].mean()),
                            'mean_interval_width': float(widths[stratum_mask].mean()),
                        })
        print(f'{target}/{pool}: calibration + conformal + AD complete for all 3 representations')
        log_milestone(target, pool, 'Module E (calibration/conformal/AD) complete')

calibration_df = pd.DataFrame(calibration_records)
conformal_df = pd.DataFrame(conformal_records)
calibration_path = results_path / shard_name('calibration_summary.csv')
conformal_path = results_path / shard_name('conformal_prediction_summary.csv')
calibration_df.to_csv(calibration_path, index=False)
conformal_df.to_csv(conformal_path, index=False)
_register('calibration_summary.csv', calibration_path, len(calibration_df))
_register('conformal_prediction_summary.csv', conformal_path, len(conformal_df))

reliability_curves_df = pd.DataFrame(reliability_curve_records)
reliability_path = results_path / shard_name('reliability_curves.csv')
reliability_curves_df.to_csv(reliability_path, index=False)
_register('reliability_curves.csv', reliability_path, len(reliability_curves_df))

venn_abers_intervals_df = pd.DataFrame(venn_abers_interval_records)
venn_abers_intervals_path = results_path / shard_name('venn_abers_intervals.csv')
venn_abers_intervals_df.to_csv(venn_abers_intervals_path, index=False)
_register('venn_abers_intervals.csv', venn_abers_intervals_path, len(venn_abers_intervals_df))

print(f'\nCalibration summary saved: {calibration_path} ({len(calibration_df)} rows)')
print(f'Conformal prediction summary saved: {conformal_path} ({len(conformal_df)} rows)')
print(f'Venn-ABERS per-compound intervals saved: {venn_abers_intervals_path} ({len(venn_abers_intervals_df)} rows)')





In [ ]:
# ---- Figure: calibration (raw vs Platt-calibrated Brier score), all
# targets, combined representation, full pool -- does calibration degrade
# for receptors with more heterogeneous assay data (Q4). ----
cal_plot_df = calibration_df[
    (calibration_df['feature_representation'] == 'combined') & (calibration_df['activity_pool'] == 'full')
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
x = np.arange(len(TARGETS))
width = 0.25
for i, algo_label in enumerate([ALGO_LABELS[a] for a in ALGORITHMS]):
    sub = cal_plot_df[cal_plot_df['algorithm'] == algo_label].set_index('target').reindex(TARGETS)
    axes[0].bar(x + (i - 1) * width, sub['brier_raw'], width, label=algo_label, color=ALGO_COLORS[ALGORITHMS[i]], alpha=0.85)
    axes[1].bar(x + (i - 1) * width, sub['ece_raw'], width, label=algo_label, color=ALGO_COLORS[ALGORITHMS[i]], alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels([t.upper() for t in TARGETS], rotation=45, ha='right')
axes[0].set_ylabel('Brier score (raw)')
panel_label(axes[0], 'A')
axes[1].set_xticks(x); axes[1].set_xticklabels([t.upper() for t in TARGETS], rotation=45, ha='right')
axes[1].set_ylabel('Expected calibration error (raw)')
axes[1].legend(frameon=False, fontsize=8)
panel_label(axes[1], 'B')
fig.tight_layout()
save_fig(fig, 'figure_calibration_by_target', fig_dir)
_register('figure_calibration_by_target.png', fig_dir / 'figure_calibration_by_target.png')
_register('figure_calibration_by_target.pdf', fig_dir / 'figure_calibration_by_target.pdf')
cal_plot_df.to_csv(fig_dir / shard_name('figure_calibration_by_target_data.csv'), index=False)
_register('figure_calibration_by_target_data.csv', fig_dir / shard_name('figure_calibration_by_target_data.csv'), len(cal_plot_df))

# ---- Figure: conformal coverage vs nominal confidence level, within-AD vs
# outside-AD -- does the applicability domain track reliability (Q4). ----
conf_plot_df = conformal_df[
    (conformal_df['feature_representation'] == 'combined') & (conformal_df['activity_pool'] == 'full') &
    (conformal_df['task'] == 'classification') & (conformal_df['stratum'].isin(['within_ad', 'outside_ad']))
]
fig, ax = plt.subplots(figsize=(5.5, 5))
for stratum, marker, color in [('within_ad', 'o', '#0072B2'), ('outside_ad', 's', '#D55E00')]:
    sub = conf_plot_df[conf_plot_df['stratum'] == stratum].groupby('confidence_level')['empirical_coverage'].mean()
    ax.plot(sub.index, sub.values, marker=marker, color=color, label=stratum.replace('_', ' '))
ax.plot([0.8, 0.95], [0.8, 0.95], 'k--', linewidth=0.8, alpha=0.6)
ax.set_xlabel('Nominal confidence level')
ax.set_ylabel('Empirical coverage')
ax.legend(frameon=False)
panel_label(ax, 'A')
fig.tight_layout()
save_fig(fig, 'figure_conformal_coverage_by_ad_status', fig_dir)
_register('figure_conformal_coverage_by_ad_status.png', fig_dir / 'figure_conformal_coverage_by_ad_status.png')
_register('figure_conformal_coverage_by_ad_status.pdf', fig_dir / 'figure_conformal_coverage_by_ad_status.pdf')
conf_plot_df.to_csv(fig_dir / shard_name('figure_conformal_coverage_by_ad_status_data.csv'), index=False)
_register('figure_conformal_coverage_by_ad_status_data.csv', fig_dir / shard_name('figure_conformal_coverage_by_ad_status_data.csv'), len(conf_plot_df))
print('Figures saved: figure_calibration_by_target, figure_conformal_coverage_by_ad_status (+ plot-source CSVs)')





## Module F: Interpretability

SHAP (`TreeExplainer`), fingerprint-bit-to-SMARTS decoding, feature-importance
stability across the repeated-CV repeats, and hyperparameter stability
across the 50 Optuna trials.

every function
below that names "the model" — SHAP, fingerprint decoding, Y-randomization
(Module H) — takes the actual best-performing model as an explicit
parameter. Y-randomization uses the deployed algorithm for each target rather than
of which algorithm actually won (LightGBM won both targets it was run on).
The best-performing algorithm per (target, pool, task) was determined at the
end of Module D by DEPLOYMENT inner-CV score (Optuna best_value on cal_train)
— never by test performance — so no single-model analysis here selects its
algorithm using data it is then evaluated on. Every module reads that one
leak-free `best_algorithm` determination rather than re-deriving its own.


In [ ]:
ALGO_LABEL_TO_KEY = {v: k for k, v in ALGO_LABELS.items()}

def run_shap_analysis(model, algo_key, X_train, X_test, feature_names, out_dir, target, pool, representation):
    '''SHAP TreeExplainer on the actual best model (passed in explicitly —
    never assumed). Persists the full SHAP value matrix and the explained
    samples alongside the summary plot, unlike the predecessor which
    deleted (.pop()) shap_values_matrix/X_explain_dense before saving.'''
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    if isinstance(shap_values, list):
        shap_values = shap_values[1] if len(shap_values) == 2 else shap_values[0]
    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1] if shap_values.shape[-1] == 2 else shap_values.mean(axis=-1)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    n_top = min(20, len(mean_abs_shap))
    top_idx = np.argsort(mean_abs_shap)[-n_top:][::-1]
    top_features = [feature_names[i] for i in top_idx]

    fig = plt.figure(figsize=(8, 6))
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=20, show=False)
    save_fig(fig, 'shap_summary', out_dir)

    np.save(out_dir / 'shap_values.npy', shap_values)
    np.save(out_dir / 'shap_X_explain.npy', X_test)
    shap_summary = {
        'target': target, 'activity_pool': pool, 'feature_representation': representation,
        'algorithm': ALGO_LABELS[algo_key],
        'top_features': top_features,
        'top_features_indices': [int(i) for i in top_idx],
        'mean_abs_shap': [float(v) for v in mean_abs_shap[top_idx]],
    }
    with open(out_dir / 'shap_summary.json', 'w') as f:
        json.dump(shap_summary, f, indent=2)
    return shap_summary

print('run_shap_analysis() defined (best model passed explicitly, full SHAP arrays persisted).')





In [ ]:
def decode_shap_fingerprint_bits(shap_summary, mols, y_class_train, train_fp_mask,
                                  radius=MORGAN_RADIUS, n_bits=MORGAN_NBITS, n_top=5, min_occurrences=5):
    '''Decode top Morgan fingerprint bits from SHAP importance to SMARTS
    substructure patterns, with enrichment ratios in actives vs inactives.
    Generalized from the predecessor (radius/n_bits are the module-level
    globals, not silently-divergent hardcoded defaults); no numbered
    figure/table references in any printed or saved text.'''
    morgan_bits = []
    for feat, score in zip(shap_summary['top_features'], shap_summary['mean_abs_shap']):
        if feat.startswith('Morgan_'):
            try:
                bit_id = int(feat.split('_')[1])
                morgan_bits.append((feat, bit_id, score))
            except (ValueError, IndexError):
                continue
    if not morgan_bits:
        return pd.DataFrame()

    def get_bit_smarts(mol, bit_id):
        bi = {}
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits, bitInfo=bi)
        if bit_id not in bi:
            return None
        atom_idx, env_radius = bi[bit_id][0]
        env = Chem.FindAtomEnvironmentOfRadiusN(mol, env_radius, atom_idx)
        if len(env) == 0:
            atom = mol.GetAtomWithIdx(atom_idx)
            return f'[{atom.GetSmarts()}]'
        submol = Chem.PathToSubmol(mol, list(env))
        return Chem.MolToSmarts(submol) if submol and submol.GetNumAtoms() > 0 else None

    train_mols = [m for m, keep in zip(mols, train_fp_mask) if keep]
    labels = y_class_train[train_fp_mask]
    total_active = int(labels.sum())
    total_inactive = int(len(labels) - total_active)

    rows = []
    for feat_name, bit_id, shap_imp in morgan_bits[:n_top]:
        train_fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius=radius, nBits=n_bits) for m in train_mols]
        bit_mask = np.array([bool(fp[bit_id]) for fp in train_fps])
        mols_with_bit = [m for m, b in zip(train_mols, bit_mask) if b]
        if len(mols_with_bit) < min_occurrences:
            continue
        smarts_list = [s for m in mols_with_bit[:30] if (s := get_bit_smarts(m, bit_id)) is not None]
        if smarts_list:
            smarts_pattern, pattern_freq = Counter(smarts_list).most_common(1)[0]
        else:
            smarts_pattern, pattern_freq = 'N/A', 0
        active_with_bit = int((bit_mask & (labels == 1)).sum())
        inactive_with_bit = int((bit_mask & (labels == 0)).sum())
        pct_active = active_with_bit / total_active * 100 if total_active > 0 else 0.0
        pct_inactive = inactive_with_bit / total_inactive * 100 if total_inactive > 0 else 0.0
        enrichment = pct_active / pct_inactive if pct_inactive > 0 else float('inf')
        rows.append({
            'bit_id': feat_name, 'mean_abs_shap': round(shap_imp, 4),
            'smarts_pattern': smarts_pattern, 'modal_pattern_count': pattern_freq,
            'n_occurrences_sampled': len(smarts_list),
            'n_actives_with_bit': active_with_bit, 'pct_actives': round(pct_active, 1),
            'n_inactives_with_bit': inactive_with_bit, 'pct_inactives': round(pct_inactive, 1),
            'enrichment_ratio': round(enrichment, 2) if enrichment != float('inf') else np.inf,
        })
    return pd.DataFrame(rows)

print('decode_shap_fingerprint_bits() defined (radius/n_bits from module-level globals).')





In [ ]:
def feature_importance_stability(importance_records, algo_label, task, top_n=20):
    '''Top-N native feature_importances_ overlap across the nested-CV outer
    folds (mean pairwise Jaccard + features present in every fold) — for
    whichever algorithm was determined best, not hardcoded.'''
    matching = [r for r in importance_records if r['algorithm'] == algo_label and r['task'] == task]
    if len(matching) < 2:
        return None
    top_sets = {r['outer_fold']: set(np.argsort(r['importances'])[-top_n:]) for r in matching}
    ids = sorted(top_sets.keys())
    jaccard_sum, n_pairs = 0.0, 0
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            a, b = top_sets[ids[i]], top_sets[ids[j]]
            jaccard_sum += len(a & b) / len(a | b)
            n_pairs += 1
    mean_jaccard = jaccard_sum / n_pairs if n_pairs else float('nan')
    in_every_repeat = set.intersection(*top_sets.values()) if top_sets else set()
    return {
        'n_repeats_with_importances': len(ids),
        'mean_pairwise_jaccard_top_n': mean_jaccard,
        'n_features_in_top_n_every_repeat': len(in_every_repeat),
        'top_n': top_n,
    }


def hyperparameter_stability(hyperparams_history_path):
    '''Analyze Optuna trial history: do the top trials' hyperparameters
    cluster, or vary wildly across the 50-trial budget? Cheap, reuses the
    already-computed Optuna trials -- no extra model fits.'''
    if not Path(hyperparams_history_path).exists():
        return None
    with open(hyperparams_history_path) as f:
        history = json.load(f)
    rows = []
    for tag, info in history.items():
        rows.append({'model_task': tag, 'best_value': info.get('best_value'), 'n_trials': info.get('n_trials')})
    return pd.DataFrame(rows)

print('feature_importance_stability() and hyperparameter_stability() defined.')





In [ ]:
# =============================================================================
# MODULE F MASTER LOOP
# =============================================================================
shap_summaries = []
smarts_tables = []
stability_records = []
hyperparam_stability_records = []

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')
        y_class = labels_df['activity'].to_numpy(int)
        y_reg = labels_df['pActivity'].to_numpy(float)
        hyperparams = all_hyperparams[cfg['combo_key']]

        # Same canonical split as Modules E/H — SHAP explains the DEPLOYED
        # model (fit on cal_train_idx), on the held-out test set.
        train_idx, test_idx, cal_train_idx, cal_holdout_idx = canonical_splits(labels_df, cfg)

        history_path = cfg['results_dir'] / 'optuna_tuning_history.json'
        hp_stab_df = hyperparameter_stability(history_path)
        if hp_stab_df is not None:
            hp_stab_df.insert(0, 'activity_pool', pool)
            hp_stab_df.insert(0, 'target', target)
            hyperparam_stability_records.append(hp_stab_df)

        # Mols computed ONCE per (target, pool), reused for whichever
        # representation(s) need fingerprint-bit decoding below (was
        # recomputed redundantly per representation before this fix).
        smiles_lookup = dict(zip(merged[(target, pool)]['global_compound_id'], merged[(target, pool)]['clean_smiles']))
        mols = smiles_to_mols(labels_df['global_compound_id'].map(smiles_lookup).tolist())
        train_fp_mask = np.zeros(len(labels_df), dtype=bool)
        train_fp_mask[cal_train_idx] = True  # the deployed model's actual fitting set

        for representation in FEATURE_REPS:
            rep_results_dir = cfg['results_dir'] / representation
            rep_results_dir.mkdir(parents=True, exist_ok=True)
            X = np.load(cfg['features_dir'] / f'{representation}.npy')
            with open(cfg['features_dir'] / f'{representation}_feature_names.json') as f:
                feature_names = json.load(f)

            best_clf_label = best_algorithm.get((target, pool, representation, 'classification'))
            if best_clf_label is None:
                continue
            best_clf_key = ALGO_LABEL_TO_KEY[best_clf_label]

            clf_params = add_task_defaults(hyperparams[f'{best_clf_key}_classification'], best_clf_key, 'classification', cfg['random_state'])
            best_clf = build_model(best_clf_key, 'classification', clf_params)
            best_clf.fit(X[cal_train_idx], y_class[cal_train_idx])

            shap_summary = run_shap_analysis(
                best_clf, best_clf_key, X[cal_train_idx], X[test_idx], feature_names,
                rep_results_dir, target, pool, representation,
            )
            shap_summaries.append(shap_summary)

            if representation in ('morgan', 'combined'):
                smarts_df = decode_shap_fingerprint_bits(shap_summary, mols, y_class, train_fp_mask)
                if len(smarts_df) > 0:
                    smarts_df.insert(0, 'feature_representation', representation)
                    smarts_df.insert(0, 'activity_pool', pool)
                    smarts_df.insert(0, 'target', target)
                    smarts_tables.append(smarts_df)

            importances_path = cfg['results_dir'] / f'nested_importances_{representation}.json'
            if importances_path.exists():
                with open(importances_path) as f:
                    importance_records = json.load(f)
                stab = feature_importance_stability(importance_records, best_clf_label, 'classification')
                if stab is not None:
                    stab.update({'target': target, 'activity_pool': pool, 'feature_representation': representation,
                                 'algorithm': best_clf_label})
                    stability_records.append(stab)

        print(f'{target}/{pool}: SHAP + fingerprint decoding + stability complete')
        log_milestone(target, pool, 'Module F (interpretability) complete')

shap_summaries_path = results_path / shard_name('shap_summaries.json')
with open(shap_summaries_path, 'w') as f:
    json.dump(shap_summaries, f, indent=2)
_register('shap_summaries.json', shap_summaries_path)

if smarts_tables:
    smarts_all_df = pd.concat(smarts_tables, ignore_index=True)
else:
    smarts_all_df = pd.DataFrame()
smarts_path = results_path / shard_name('shap_smarts_patterns.csv')
smarts_all_df.to_csv(smarts_path, index=False)
_register('shap_smarts_patterns.csv', smarts_path, len(smarts_all_df))

stability_df = pd.DataFrame(stability_records)
stability_path = results_path / shard_name('feature_importance_stability.csv')
stability_df.to_csv(stability_path, index=False)
_register('feature_importance_stability.csv', stability_path, len(stability_df))

if hyperparam_stability_records:
    hp_stability_df = pd.concat(hyperparam_stability_records, ignore_index=True)
else:
    hp_stability_df = pd.DataFrame()
hp_stability_path = results_path / shard_name('hyperparameter_stability.csv')
hp_stability_df.to_csv(hp_stability_path, index=False)
_register('hyperparameter_stability.csv', hp_stability_path, len(hp_stability_df))

print(f'\nSHAP summaries: {len(shap_summaries)}')
print(f'SMARTS patterns decoded: {len(smarts_all_df)}')
print(f'Feature-importance stability rows: {len(stability_df)}')
print(f'Hyperparameter stability rows: {len(hp_stability_df)}')





In [ ]:
# ---- Hyperparameter VALUE stability across targets/pools: what did Optuna
# actually choose (best_depth, best_learning_rate, best_lambda, ...) for each
# model, and how consistent is that across the 5 targets? Reads the tuned
# best_params already in memory — no extra model fits. ----
hp_value_rows = []
for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        hp = all_hyperparams.get(cfg['combo_key'], {})
        for model_task, params in hp.items():
            if not isinstance(params, dict):
                continue
            for param_name, param_value in params.items():
                hp_value_rows.append({
                    'target': target, 'activity_pool': pool, 'model_task': model_task,
                    'hyperparameter': param_name, 'value': param_value,
                })

hp_values_df = pd.DataFrame(hp_value_rows)
hp_values_path = results_path / shard_name('hyperparameter_values_by_combination.csv')
hp_values_df.to_csv(hp_values_path, index=False)
_register('hyperparameter_values_by_combination.csv', hp_values_path, len(hp_values_df))

# Cross-target spread of each numeric hyperparameter per model_task (how much
# does the chosen value move across targets — did XGBoost always pick depth ~6,
# or swing 3-15?).
if len(hp_values_df):
    hp_numeric = hp_values_df.copy()
    hp_numeric['value_numeric'] = pd.to_numeric(hp_numeric['value'], errors='coerce')
    hp_spread = (
        hp_numeric.dropna(subset=['value_numeric'])
        .groupby(['model_task', 'hyperparameter'])['value_numeric']
        .agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
    )
    hp_spread_path = results_path / shard_name('hyperparameter_spread_across_targets.csv')
    hp_spread.to_csv(hp_spread_path, index=False)
    _register('hyperparameter_spread_across_targets.csv', hp_spread_path, len(hp_spread))
    print(f'Hyperparameter value table: {len(hp_values_df)} rows -> {hp_values_path}')
    print(f'Cross-target spread: {len(hp_spread)} (model_task x hyperparameter) rows -> {hp_spread_path}')





In [ ]:
# ---- Optuna convergence: does the running-best objective plateau before the
# 50-trial budget is exhausted (i.e. were 50 trials enough)? Averages the
# running-best trajectory across all persisted studies per (algorithm, task).
# Reads the per-trial CSVs persisted during Module D tuning — no re-tuning. ----
def _parse_optuna_trial_filename(tpath):
    # Return (algo, task) from either old or study-prefixed trial CSV names.
    stem = tpath.stem
    if not stem.startswith('optuna_trials_'):
        return None, None
    parts = stem.replace('optuna_trials_', '', 1).split('_')
    if len(parts) < 2:
        return None, None
    algo, task = parts[-2], parts[-1]
    if algo not in ALGORITHMS or task not in TASKS:
        return None, None
    return algo, task


convergence_frames = []
for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        # Current files are optuna_trials_{study_prefix}_{algo}_{task}.csv;
        # the parser also accepts the earlier optuna_trials_{algo}_{task}.csv
        # convention so reruns remain backward-compatible.
        for tpath in sorted(cfg['results_dir'].glob('optuna_trials_*.csv')):
            algo, task = _parse_optuna_trial_filename(tpath)
            if algo is None:
                continue
            tdf = pd.read_csv(tpath)
            if 'value' not in tdf.columns or 'number' not in tdf.columns:
                continue
            tdf = tdf[['number', 'value']].dropna().sort_values('number')
            if tdf.empty:
                continue
            running_best = tdf['value'].cummax().to_numpy()
            convergence_frames.append(pd.DataFrame({
                'target': target,
                'activity_pool': pool,
                'study_file': tpath.name,
                'algorithm': ALGO_LABELS[algo],
                'task': task,
                'trial': np.arange(1, len(running_best) + 1),
                'running_best': running_best,
            }))

if convergence_frames:
    convergence_long = pd.concat(convergence_frames, ignore_index=True)
    convergence_mean = (
        convergence_long.groupby(['algorithm', 'task', 'trial'])['running_best']
        .agg(mean='mean', std='std', n_studies='count')
        .reset_index()
    )
    convergence_mean['std'] = convergence_mean['std'].fillna(0.0)

    convergence_path = results_path / shard_name('optuna_convergence.csv')
    convergence_mean.to_csv(convergence_path, index=False)
    _register('optuna_convergence.csv', convergence_path, len(convergence_mean))

    convergence_long_path = results_path / shard_name('optuna_convergence_long.csv')
    convergence_long.to_csv(convergence_long_path, index=False)
    _register('optuna_convergence_long.csv', convergence_long_path, len(convergence_long))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
    for ax, task, ylabel, letter in [
        (axes[0], 'classification', 'Running-best CV ROC-AUC', 'A'),
        (axes[1], 'regression', 'Running-best CV R²', 'B'),
    ]:
        task_conv = convergence_mean[convergence_mean['task'] == task]
        for algo in ALGORITHMS:
            sub = task_conv[task_conv['algorithm'] == ALGO_LABELS[algo]].sort_values('trial')
            if sub.empty:
                continue
            x = sub['trial'].to_numpy(dtype=float)
            y = sub['mean'].to_numpy(dtype=float)
            sd = sub['std'].to_numpy(dtype=float)
            ax.plot(x, y, color=ALGO_COLORS[algo], label=ALGO_LABELS[algo])
            ax.fill_between(x, y - sd, y + sd, color=ALGO_COLORS[algo], alpha=0.15, linewidth=0)
        ax.set_xlabel('Optuna trial')
        ax.set_ylabel(ylabel)
        panel_label(ax, letter)
    axes[0].legend(frameon=False)
    fig.tight_layout()
    save_fig(fig, 'figure_optuna_convergence', fig_dir)
    _register('figure_optuna_convergence.png', fig_dir / 'figure_optuna_convergence.png')
    _register('figure_optuna_convergence.pdf', fig_dir / 'figure_optuna_convergence.pdf')
    print(f'Optuna convergence saved: {convergence_path} ({len(convergence_long)} trial rows across '
          f'{convergence_long["study_file"].nunique()} study files)')
else:
    print('No Optuna trial CSVs found — convergence skipped.')








## Module G: Statistical Benchmarking

Wilcoxon signed-rank tests between algorithm pairs (paired by repeat index
— same train/test split per repeat, genuine matched pairs, reusing the
full model names), bootstrap
confidence intervals for every headline metric, and effect sizes reported
alongside significance (lesson 24 — "significant" is not "large";
rank-biserial correlation is reported next to every Wilcoxon p-value).
Computed immediately here for every (target, pool) combination, not
deferred to write-up.


In [ ]:
def rank_biserial_effect_size(x, y):
    '''Matched-pairs rank-biserial correlation: effect size for the Wilcoxon
    signed-rank test. r = (n_positive - n_negative) / n_total differences
    (ties excluded), in [-1, 1] — the standard paired-test companion effect
    size to accompany a Wilcoxon p-value (lesson 24: report magnitude, not
    just significance).'''
    diffs = np.asarray(x) - np.asarray(y)
    diffs = diffs[diffs != 0]
    if len(diffs) == 0:
        return 0.0
    n_pos = int((diffs > 0).sum())
    n_neg = int((diffs < 0).sum())
    return (n_pos - n_neg) / len(diffs)


def bootstrap_ci(values, n_bootstrap=1000, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    values = np.asarray(values, dtype=float)
    n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean() for _ in range(n_bootstrap)])
    return {
        'mean': float(values.mean()), 'std': float(values.std()),
        'ci_lo': float(np.percentile(boot_means, 2.5)), 'ci_hi': float(np.percentile(boot_means, 97.5)),
    }

print('Effect-size and bootstrap-CI helper functions defined.')





In [ ]:
# =============================================================================
# MODULE G MASTER LOOP — paired Wilcoxon + effect size + bootstrap CI, per
# (target, pool, representation, task, metric), across every algorithm pair.
# Pairs are matched by OUTER FOLD (same outer-train/outer-test per fold, so
# genuinely paired). With N_OUTER_FOLDS folds the paired Wilcoxon has small n
# and a correspondingly high p-value floor — effect sizes (rank-biserial) and
# bootstrap CIs are reported alongside and should carry the interpretation.
# =============================================================================
statistical_comparison_records = []
bootstrap_ci_records = []
ALGO_PAIRS = [('Random Forest', 'XGBoost'), ('Random Forest', 'LightGBM'), ('XGBoost', 'LightGBM')]

for (target, pool, representation), combo_df in benchmark_results_long.groupby(
        ['target', 'activity_pool', 'feature_representation']):
    for task, metric_name in [('classification', 'roc_auc'), ('regression', 'r2')]:
        task_df = combo_df[(combo_df['task'] == task) & (combo_df['metric_name'] == metric_name)]
        pivot = task_df.pivot(index='outer_fold', columns='algorithm', values='metric_value')

        for model_a, model_b in ALGO_PAIRS:
            if model_a not in pivot.columns or model_b not in pivot.columns:
                continue
            vals_a, vals_b = pivot[model_a].values, pivot[model_b].values
            if len(vals_a) < 2 or np.allclose(vals_a, vals_b):
                continue
            try:
                stat, p_val = wilcoxon(vals_a, vals_b)
            except ValueError:
                continue
            effect = rank_biserial_effect_size(vals_a, vals_b)
            statistical_comparison_records.append({
                'target': target, 'activity_pool': pool, 'feature_representation': representation,
                'task': task, 'metric': metric_name, 'model_a': model_a, 'model_b': model_b,
                'model_a_mean': float(vals_a.mean()), 'model_b_mean': float(vals_b.mean()),
                'mean_difference': float(vals_a.mean() - vals_b.mean()),
                'wilcoxon_statistic': float(stat), 'wilcoxon_p': float(p_val),
                'rank_biserial_effect_size': float(effect),
                'higher_mean_model': model_a if vals_a.mean() > vals_b.mean() else model_b,
            })

        for algo in pivot.columns:
            ci = bootstrap_ci(pivot[algo].values)
            ci.update({'target': target, 'activity_pool': pool, 'feature_representation': representation,
                       'task': task, 'metric': metric_name, 'algorithm': algo})
            bootstrap_ci_records.append(ci)

statistical_comparisons_df = pd.DataFrame(statistical_comparison_records)
bootstrap_ci_df = pd.DataFrame(bootstrap_ci_records)
stat_comparisons_path = results_path / shard_name('statistical_comparisons.csv')
bootstrap_ci_path = results_path / shard_name('bootstrap_confidence_intervals.csv')
statistical_comparisons_df.to_csv(stat_comparisons_path, index=False)
bootstrap_ci_df.to_csv(bootstrap_ci_path, index=False)
_register('statistical_comparisons.csv', stat_comparisons_path, len(statistical_comparisons_df))
_register('bootstrap_confidence_intervals.csv', bootstrap_ci_path, len(bootstrap_ci_df))

print(f'Statistical comparisons (Wilcoxon + effect size): {len(statistical_comparisons_df)} rows -> {stat_comparisons_path}')
print(f'Bootstrap confidence intervals: {len(bootstrap_ci_df)} rows -> {bootstrap_ci_path}')





## Module H: Final Models, Error Analysis, and Everything Persisted

Train final models (best hyperparameters from Module D) on the canonical
scaffold split's training partition for every (target, pool, representation)
x algorithm x task, and save every artifact a later notebook would need to
reuse this work without retraining: trained model files, feature
names/order, hyperparameters, calibration/conformal/AD objects (already
saved in Module E), and train/test compound IDs using the
`global_compound_id`/`global_scaffold_id` convention from notebook 02.

Also: Y-randomization (fixed to take the actual best model, not a hardcoded
algorithm), learning curves, persistent-misclassification error analysis,
scaffold-level performance breakdown (singleton vs non-singleton scaffolds
— directly answers Q2), and prediction-difficulty analysis (error vs MW,
LogP, pActivity, n_measurements, scaffold_size — also Q2).


In [ ]:
final_model_records = []
timing_records = []          # tune/train/predict seconds per algorithm
prediction_export_rows = []  # per-compound deployed-model test predictions
agreement_records = []       # RF/XGBoost/LightGBM consensus vs disagreement

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')
        y_class = labels_df['activity'].to_numpy(int)
        y_reg = labels_df['pActivity'].to_numpy(float)
        hyperparams = all_hyperparams[cfg['combo_key']]

        # Deployed models are fit on cal_train_idx (calibration holdout reserved),
        # so a saved model exactly matches its Module-E calibrator/conformal wrapper.
        train_idx, test_idx, cal_train_idx, cal_holdout_idx = canonical_splits(labels_df, cfg)
        assert_no_scaffold_leakage(labels_df, train_idx, test_idx)
        assert_no_scaffold_leakage(labels_df, cal_train_idx, test_idx)

        # Three-way split saved so downstream (screening) knows exactly what the
        # deployed model saw: cal_train (fit) / cal_holdout (calibration) / test.
        split_of = {}
        for j in cal_train_idx:   split_of[int(j)] = 'cal_train'
        for j in cal_holdout_idx: split_of[int(j)] = 'cal_holdout'
        for j in test_idx:        split_of[int(j)] = 'test'
        split_records = [
            {'global_compound_id': labels_df['global_compound_id'].iloc[j], 'split': split_of[int(j)]}
            for j in sorted(split_of)
        ]
        split_path = cfg['results_dir'] / 'train_test_split.csv'
        pd.DataFrame(split_records).to_csv(split_path, index=False)
        np.save(cfg['results_dir'] / 'cal_train_indices.npy', cal_train_idx)
        np.save(cfg['results_dir'] / 'cal_holdout_indices.npy', cal_holdout_idx)
        np.save(cfg['results_dir'] / 'test_indices.npy', test_idx)

        for representation in FEATURE_REPS:
            rep_models_dir = cfg['models_dir'] / representation
            rep_models_dir.mkdir(parents=True, exist_ok=True)
            X = np.load(cfg['features_dir'] / f'{representation}.npy')
            with open(cfg['features_dir'] / f'{representation}_feature_names.json') as f:
                feature_names = json.load(f)

            reg_train_mask = ~np.isnan(y_reg[cal_train_idx])
            reg_test_mask = ~np.isnan(y_reg[test_idx])

            # Applicability-domain status for every test compound (from the
            # frozen Module-E artifact — computed once, never refit here).
            ad_ref_path = rep_models_dir / 'ad_reference.pkl'
            if ad_ref_path.exists():
                ad_ref = joblib.load(ad_ref_path)
                test_ad_dist, test_within_ad = apply_applicability_domain(ad_ref, X[test_idx])
            else:
                test_ad_dist = np.full(len(test_idx), np.nan)
                test_within_ad = np.full(len(test_idx), False)

            clf_proba_by_algo = {}
            for algo in ALGORITHMS:
                clf_params = add_task_defaults(hyperparams[f'{algo}_classification'], algo, 'classification', cfg['random_state'])
                clf = build_model(algo, 'classification', clf_params)
                _t0 = time.time(); clf.fit(X[cal_train_idx], y_class[cal_train_idx]); clf_fit_s = time.time() - _t0
                _t0 = time.time(); clf_proba = clf.predict_proba(X[test_idx])[:, 1]; clf_pred_s = time.time() - _t0
                clf_auc = roc_auc_score(y_class[test_idx], clf_proba)
                joblib.dump(clf, rep_models_dir / f'{algo}_clf.joblib')
                clf_proba_by_algo[algo] = clf_proba

                reg_params = add_task_defaults(hyperparams[f'{algo}_regression'], algo, 'regression', cfg['random_state'])
                reg = build_model(algo, 'regression', reg_params)
                _t0 = time.time(); reg.fit(X[cal_train_idx][reg_train_mask], y_reg[cal_train_idx][reg_train_mask]); reg_fit_s = time.time() - _t0
                reg_pred = reg.predict(X[test_idx][reg_test_mask])
                reg_r2 = r2_score(y_reg[test_idx][reg_test_mask], reg_pred)
                joblib.dump(reg, rep_models_dir / f'{algo}_reg.joblib')

                # Full-length regression prediction vector (NaN where pActivity
                # was missing) so it lines up with the per-compound export.
                reg_pred_full = np.full(len(test_idx), np.nan)
                reg_pred_full[reg_test_mask] = reg_pred

                final_model_records.append({
                    'target': target, 'activity_pool': pool, 'feature_representation': representation,
                    'algorithm': ALGO_LABELS[algo], 'test_roc_auc': float(clf_auc), 'test_r2': float(reg_r2),
                })
                timing_records.append({
                    'target': target, 'activity_pool': pool, 'feature_representation': representation,
                    'algorithm': ALGO_LABELS[algo],
                    'classifier_fit_seconds': clf_fit_s, 'classifier_predict_seconds': clf_pred_s,
                    'regressor_fit_seconds': reg_fit_s,
                })  # Optuna tuning time is recorded separately in optuna_tuning_history.json

                # Per-compound deployed-model test predictions (this algorithm).
                for row_i, test_i in enumerate(test_idx):
                    prediction_export_rows.append({
                        'target': target, 'activity_pool': pool, 'feature_representation': representation,
                        'algorithm': ALGO_LABELS[algo],
                        'global_compound_id': labels_df['global_compound_id'].iloc[test_i],
                        'molecule_chembl_id': labels_df['molecule_chembl_id'].iloc[test_i],
                        'y_true_class': int(y_class[test_i]),
                        'predicted_proba': float(clf_proba[row_i]),
                        'predicted_class': int(clf_proba[row_i] >= 0.5),
                        'y_true_pactivity': float(y_reg[test_i]) if not np.isnan(y_reg[test_i]) else np.nan,
                        'predicted_pactivity': float(reg_pred_full[row_i]) if not np.isnan(reg_pred_full[row_i]) else np.nan,
                        'within_applicability_domain': bool(test_within_ad[row_i]),
                        'applicability_domain_distance': float(test_ad_dist[row_i]) if not np.isnan(test_ad_dist[row_i]) else np.nan,
                    })

            # ---- Algorithm agreement: do RF / XGBoost / LightGBM agree per
            # compound? Disagreement localizes where predictive uncertainty
            # comes from model choice rather than the compound being easy. ----
            preds_matrix = np.vstack([(clf_proba_by_algo[a] >= 0.5).astype(int) for a in ALGORITHMS])  # (3, n_test)
            proba_matrix = np.vstack([clf_proba_by_algo[a] for a in ALGORITHMS])
            for row_i, test_i in enumerate(test_idx):
                votes = preds_matrix[:, row_i]
                all_agree = bool(len(set(votes.tolist())) == 1)
                agreement_records.append({
                    'target': target, 'activity_pool': pool, 'feature_representation': representation,
                    'global_compound_id': labels_df['global_compound_id'].iloc[test_i],
                    'y_true_class': int(y_class[test_i]),
                    'n_algorithms_predicting_active': int(votes.sum()),
                    'all_algorithms_agree': all_agree,
                    'consensus_proba': float(proba_matrix[:, row_i].mean()),
                    'consensus_class': int(round(votes.mean())),
                })

            with open(rep_models_dir / 'feature_names.json', 'w') as f:
                json.dump(feature_names, f)
            with open(rep_models_dir / 'hyperparameters_used.json', 'w') as f:
                json.dump(hyperparams, f, indent=2)

        print(f'{target}/{pool}: final models trained + persisted for all 3 representations x 3 algorithms x 2 tasks')
        log_milestone(target, pool, 'Module H final model training complete')

final_models_df = pd.DataFrame(final_model_records)
final_models_path = results_path / shard_name('final_model_test_performance.csv')
final_models_df.to_csv(final_models_path, index=False)
_register('final_model_test_performance.csv', final_models_path, len(final_models_df))
print(f'\nFinal model test performance saved: {final_models_path} ({len(final_models_df)} rows)')

# Per-compound deployed-model test predictions (the single most useful rerun
# safeguard — investigate any unexpected result without retraining).
predictions_df = pd.DataFrame(prediction_export_rows)
predictions_path = results_path / shard_name('deployed_model_test_predictions.csv')
predictions_df.to_csv(predictions_path, index=False)
_register('deployed_model_test_predictions.csv', predictions_path, len(predictions_df))
print(f'Per-compound test predictions saved: {predictions_path} ({len(predictions_df)} rows)')

agreement_df = pd.DataFrame(agreement_records)
agreement_path = results_path / shard_name('algorithm_agreement.csv')
agreement_df.to_csv(agreement_path, index=False)
_register('algorithm_agreement.csv', agreement_path, len(agreement_df))
if len(agreement_df):
    pct_unanimous = agreement_df['all_algorithms_agree'].mean() * 100
    print(f'Algorithm agreement saved: {agreement_path} ({len(agreement_df)} rows; '
          f'{pct_unanimous:.1f}% of compound-predictions unanimous across RF/XGBoost/LightGBM)')

timing_df = pd.DataFrame(timing_records)
timing_path = results_path / shard_name('timing_summary.csv')
timing_df.to_csv(timing_path, index=False)
_register('timing_summary.csv', timing_path, len(timing_df))
print(f'Timing summary saved: {timing_path} ({len(timing_df)} rows)')





In [ ]:
def run_y_randomization(model_builder_fn, algo_key, X_train, y_class_train, groups, n_iterations, cv_folds, random_state):
    '''Y-randomization: permute activity labels, retrain, confirm real
    performance exceeds chance. Takes the ACTUAL best algorithm as a
    parameter via model_builder_fn/algo_key -- bug fix for lesson 1 (the
    predecessor hardcoded XGBoost regardless of which algorithm actually won,
    which was LightGBM for both targets it was run on).'''
    rng = np.random.default_rng(random_state)
    rand_scores = []
    cv = GroupKFold(n_splits=min(cv_folds, len(np.unique(groups))))
    for _ in range(n_iterations):
        y_shuffled = rng.permutation(y_class_train)
        model = model_builder_fn()
        try:
            scores = cross_val_score(model, X_train, y_shuffled, cv=cv, groups=groups, scoring='roc_auc', n_jobs=N_CORES, error_score=0.5)
            rand_scores.append(float(scores.mean()))
        except Exception:
            rand_scores.append(0.5)

    original_model = model_builder_fn()
    orig_scores = cross_val_score(original_model, X_train, y_class_train, cv=cv, groups=groups, scoring='roc_auc', n_jobs=N_CORES)
    original_auc = float(orig_scores.mean())

    rand_mean, rand_std = float(np.mean(rand_scores)), float(np.std(rand_scores))
    z_score = (original_auc - rand_mean) / (rand_std + 1e-8)
    significance = 'p < 0.001' if z_score > 3.09 else ('p < 0.05' if z_score > 1.96 else 'not significant (p >= 0.05)')
    return {
        'algorithm': ALGO_LABELS[algo_key], 'n_iterations': n_iterations, 'cv_folds': cv.get_n_splits(),
        'original_auc': original_auc, 'randomized_mean': rand_mean, 'randomized_std': rand_std,
        'randomized_max': float(np.max(rand_scores)), 'z_score': float(z_score), 'significance': significance,
    }

print('run_y_randomization() defined (takes best algorithm as an explicit parameter).')





In [ ]:
# =============================================================================
# Y-RANDOMIZATION — run once per (target, pool) using the BEST classifier
# for the 'combined' representation (the representation used for the
# primary reported model), never a hardcoded algorithm.
# =============================================================================
y_randomization_records = []

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')
        y_class = labels_df['activity'].to_numpy(int)
        hyperparams = all_hyperparams[cfg['combo_key']]
        train_idx, _ = scaffold_train_test_split(labels_df, cfg['test_size'], cfg['random_state'])
        groups_train = get_scaffold_groups(labels_df.iloc[train_idx].reset_index(drop=True))

        X = np.load(cfg['features_dir'] / 'combined.npy')
        best_clf_label = best_algorithm.get((target, pool, 'combined', 'classification'))
        best_clf_key = ALGO_LABEL_TO_KEY[best_clf_label]
        clf_params = add_task_defaults(hyperparams[f'{best_clf_key}_classification'], best_clf_key, 'classification', cfg['random_state'])

        result = run_y_randomization(
            lambda: build_model(best_clf_key, 'classification', clf_params),
            best_clf_key, X[train_idx], y_class[train_idx], groups_train,
            n_iterations=Y_RAND_ITERATIONS, cv_folds=5, random_state=cfg['random_state'],
        )
        result.update({'target': target, 'activity_pool': pool, 'feature_representation': 'combined'})
        y_randomization_records.append(result)
        print(f'{target}/{pool}: Y-randomization on {best_clf_label} -> z={result["z_score"]:.2f} ({result["significance"]})')

y_randomization_df = pd.DataFrame(y_randomization_records)
y_rand_path = results_path / shard_name('y_randomization_summary.csv')
y_randomization_df.to_csv(y_rand_path, index=False)
_register('y_randomization_summary.csv', y_rand_path, len(y_randomization_df))
print(f'\nY-randomization summary saved: {y_rand_path}')





In [ ]:
# =============================================================================
# LEARNING CURVES — training-set-size vs performance, best classifier per
# (target, pool), combined representation, canonical scaffold test set held
# fixed while the training subset size varies.
#
# SCOPE: this is a canonical-model DIAGNOSTIC (does more training data still
# help each target?), NOT part of the unbiased nested-CV headline benchmark.
# It uses the deployment-selected algorithm + deployment hyperparameters
# (chosen by leak-free inner-CV on cal_train) and evaluates on the canonical
# test set — which those never saw, so the numbers are valid held-out
# estimates, but they are single-split points, not the nested-CV distribution.
# Report/interpret them as a data-efficiency curve, not as the primary
# generalization estimate (which is Module D's nested-CV result).
# =============================================================================
learning_curve_records = []
TRAIN_FRACTIONS = [0.1, 0.25, 0.5, 0.75, 1.0]

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')
        y_class = labels_df['activity'].to_numpy(int)
        hyperparams = all_hyperparams[cfg['combo_key']]
        train_idx, test_idx = scaffold_train_test_split(labels_df, cfg['test_size'], cfg['random_state'])

        X = np.load(cfg['features_dir'] / 'combined.npy')
        best_clf_label = best_algorithm.get((target, pool, 'combined', 'classification'))
        best_clf_key = ALGO_LABEL_TO_KEY[best_clf_label]
        clf_params = add_task_defaults(hyperparams[f'{best_clf_key}_classification'], best_clf_key, 'classification', cfg['random_state'])

        rng = np.random.default_rng(cfg['random_state'])
        for frac in TRAIN_FRACTIONS:
            n_sub = max(10, int(len(train_idx) * frac))
            sub_idx = rng.choice(train_idx, size=min(n_sub, len(train_idx)), replace=False)
            if len(np.unique(y_class[sub_idx])) < 2:
                # A too-small/imbalanced subsample (small train fractions on
                # already-small pools such as CCR5) can land single-class.
                # A model with one class has no 'active' probability to
                # report, so skip this point rather than crash or fabricate
                # a value — disclosed via the printed warning, not silent.
                print(f'  Learning curve: skipping {target}/{pool} frac={frac} '
                      f'(single-class subsample, n={len(sub_idx)})')
                continue
            model = build_model(best_clf_key, 'classification', clf_params)
            model.fit(X[sub_idx], y_class[sub_idx])
            proba = model.predict_proba(X[test_idx])[:, 1]
            learning_curve_records.append({
                'target': target, 'activity_pool': pool, 'algorithm': best_clf_label,
                'train_fraction': frac, 'n_train': len(sub_idx),
                'test_roc_auc': float(roc_auc_score(y_class[test_idx], proba)),
            })

learning_curve_df = pd.DataFrame(learning_curve_records)
learning_curve_path = results_path / shard_name('learning_curves.csv')
learning_curve_df.to_csv(learning_curve_path, index=False)
_register('learning_curves.csv', learning_curve_path, len(learning_curve_df))
print(f'Learning curves saved: {learning_curve_path} ({len(learning_curve_df)} rows)')





In [ ]:
# =============================================================================
# PERSISTENT-MISCLASSIFICATION ERROR ANALYSIS — aggregate the per-compound
# predictions emitted by Module D's leak-free nested CV. Each prediction comes
# from a model whose hyperparameters were tuned only on that outer fold's
# training scaffolds. To avoid reintroducing algorithm-selection leakage, this
# analysis does NOT filter to the deployment-selected best algorithm; it
# aggregates all three classifiers for the combined representation.
# =============================================================================
persistent_error_records = []

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')

        pred_sub = nested_predictions_df[
            (nested_predictions_df['target'] == target) &
            (nested_predictions_df['activity_pool'] == pool) &
            (nested_predictions_df['feature_representation'] == 'combined')
        ].copy()
        if pred_sub.empty:
            continue

        grouped = (
            pred_sub.groupby(['global_compound_id', 'molecule_chembl_id', 'y_true_class'])
            .agg(n_times_tested=('is_misclassified', 'size'),
                 n_times_misclassified=('is_misclassified', 'sum'),
                 mean_predicted_proba=('predicted_proba', 'mean'),
                 n_algorithms=('algorithm', 'nunique'))
            .reset_index()
        )
        grouped['misclassification_rate'] = grouped['n_times_misclassified'] / grouped['n_times_tested']
        grouped = grouped.merge(
            labels_df[['global_compound_id', 'activity', 'pActivity', 'scaffold_size', 'is_singleton_scaffold']],
            on='global_compound_id', how='left',
        )
        grouped.insert(0, 'algorithm', 'All classifiers (nested CV)')
        grouped.insert(0, 'activity_pool', pool)
        grouped.insert(0, 'target', target)
        persistent_error_records.append(grouped)
        print(f'{target}/{pool}: leak-free nested-CV error tracking complete ({len(grouped)} compounds tested)')

persistent_errors_df = pd.concat(persistent_error_records, ignore_index=True)
persistent_errors_path = results_path / shard_name('persistent_errors.csv')
persistent_errors_df.to_csv(persistent_errors_path, index=False)
_register('persistent_errors.csv', persistent_errors_path, len(persistent_errors_df))

persistently_wrong = persistent_errors_df[
    (persistent_errors_df['n_times_tested'] >= 3) & (persistent_errors_df['misclassification_rate'] >= 0.8)
]
print(f'\nPersistent errors saved: {persistent_errors_path} ({len(persistent_errors_df)} rows)')
print(f'Compounds misclassified in >=80% of leak-free nested-CV classifier test appearances (n_times_tested>=3): {len(persistently_wrong)}')






In [ ]:
# =============================================================================
# SCAFFOLD-LEVEL PERFORMANCE BREAKDOWN — AUC/R2 stratified by singleton vs
# non-singleton scaffold membership, per (target, pool, representation).
# Directly answers Q2 (does scaffold diversity predict performance).
# =============================================================================
scaffold_breakdown_records = []

for target, pool in RUN_COMBOS:
        cfg = get_config(target, pool)
        labels_df = pd.read_csv(cfg['features_dir'] / 'labels.csv')
        y_class = labels_df['activity'].to_numpy(int)
        y_reg = labels_df['pActivity'].to_numpy(float)
        hyperparams = all_hyperparams[cfg['combo_key']]
        # Deployed model (fit on cal_train_idx) evaluated on test strata.
        train_idx, test_idx, cal_train_idx, cal_holdout_idx = canonical_splits(labels_df, cfg)
        singleton_test = labels_df['is_singleton_scaffold'].to_numpy()[test_idx]

        for representation in FEATURE_REPS:
            X = np.load(cfg['features_dir'] / f'{representation}.npy')
            best_clf_label = best_algorithm.get((target, pool, representation, 'classification'))
            best_reg_label = best_algorithm.get((target, pool, representation, 'regression'))
            if best_clf_label is None or best_reg_label is None:
                continue
            clf_key, reg_key = ALGO_LABEL_TO_KEY[best_clf_label], ALGO_LABEL_TO_KEY[best_reg_label]

            clf_params = add_task_defaults(hyperparams[f'{clf_key}_classification'], clf_key, 'classification', cfg['random_state'])
            clf = build_model(clf_key, 'classification', clf_params)
            clf.fit(X[cal_train_idx], y_class[cal_train_idx])
            clf_proba = clf.predict_proba(X[test_idx])[:, 1]

            reg_train_mask = ~np.isnan(y_reg[cal_train_idx])
            reg_test_mask = ~np.isnan(y_reg[test_idx])
            reg_params = add_task_defaults(hyperparams[f'{reg_key}_regression'], reg_key, 'regression', cfg['random_state'])
            reg = build_model(reg_key, 'regression', reg_params)
            reg.fit(X[cal_train_idx][reg_train_mask], y_reg[cal_train_idx][reg_train_mask])
            reg_pred = reg.predict(X[test_idx][reg_test_mask])

            for stratum_name, mask in [('overall', np.ones(len(test_idx), dtype=bool)),
                                        ('singleton_scaffold', singleton_test),
                                        ('non_singleton_scaffold', ~singleton_test)]:
                if mask.sum() < 2:
                    continue
                row = {
                    'target': target, 'activity_pool': pool, 'feature_representation': representation,
                    'scaffold_stratum': stratum_name, 'n_compounds': int(mask.sum()),
                }
                if len(np.unique(y_class[test_idx][mask])) > 1:
                    row['classification_roc_auc'] = float(roc_auc_score(y_class[test_idx][mask], clf_proba[mask]))
                scaffold_breakdown_records.append(row)

            reg_singleton = singleton_test[reg_test_mask]
            for stratum_name, mask in [('overall', np.ones(reg_test_mask.sum(), dtype=bool)),
                                        ('singleton_scaffold', reg_singleton),
                                        ('non_singleton_scaffold', ~reg_singleton)]:
                if mask.sum() < 2:
                    continue
                scaffold_breakdown_records.append({
                    'target': target, 'activity_pool': pool, 'feature_representation': representation,
                    'scaffold_stratum': stratum_name, 'n_compounds': int(mask.sum()),
                    'regression_r2': float(r2_score(y_reg[test_idx][reg_test_mask][mask], reg_pred[mask])),
                })
        print(f'{target}/{pool}: scaffold-level breakdown complete')

scaffold_breakdown_df = pd.DataFrame(scaffold_breakdown_records)
scaffold_breakdown_path = results_path / shard_name('scaffold_performance_breakdown.csv')
scaffold_breakdown_df.to_csv(scaffold_breakdown_path, index=False)
_register('scaffold_performance_breakdown.csv', scaffold_breakdown_path, len(scaffold_breakdown_df))
print(f'\nScaffold-level performance breakdown saved: {scaffold_breakdown_path} ({len(scaffold_breakdown_df)} rows)')





In [ ]:
# =============================================================================
# DIFFICULTY ANALYSIS — prediction error vs MW / LogP / pActivity /
# n_measurements / scaffold_size (Q2: what makes a compound hard to predict).
# Reuses persistent_errors_df's per-compound misclassification_rate joined
# back to the cleaned dataset's descriptors and n_measurements.
# =============================================================================
DIFFICULTY_PREDICTORS = ['MW', 'LogP', 'pActivity', 'n_measurements', 'scaffold_size']
difficulty_records = []

for target, pool in RUN_COMBOS:
        cleaned_df = merged[(target, pool)]
        errors_sub = persistent_errors_df[
            (persistent_errors_df['target'] == target) & (persistent_errors_df['activity_pool'] == pool)
        ]
        joined = errors_sub.merge(
            cleaned_df[['global_compound_id', 'MW', 'LogP', 'n_measurements']],
            on='global_compound_id', how='left',
        )
        for predictor in DIFFICULTY_PREDICTORS:
            if predictor not in joined.columns or joined[predictor].isna().all():
                continue
            valid = joined[[predictor, 'misclassification_rate']].dropna()
            if len(valid) < 5:
                continue
            rho, p_val = spearmanr(valid[predictor], valid['misclassification_rate'])
            difficulty_records.append({
                'target': target, 'activity_pool': pool, 'predictor': predictor,
                'n_compounds': len(valid), 'spearman_r': float(rho), 'spearman_p': float(p_val),
            })

difficulty_df = pd.DataFrame(difficulty_records)
difficulty_path = results_path / shard_name('difficulty_analysis.csv')
difficulty_df.to_csv(difficulty_path, index=False)
_register('difficulty_analysis.csv', difficulty_path, len(difficulty_df))
print(f'Difficulty analysis (error vs compound/data properties) saved: {difficulty_path} ({len(difficulty_df)} rows)')





In [ ]:
# ---- Figure: learning curves, all 5 targets, full pool ----
fig, ax = plt.subplots(figsize=(6, 5))
lc_full = learning_curve_df[learning_curve_df['activity_pool'] == 'full']
for target in TARGETS:
    sub = lc_full[lc_full['target'] == target].sort_values('train_fraction')
    ax.plot(sub['n_train'], sub['test_roc_auc'], marker='o', color=TARGET_COLORS[target], label=target.upper())
ax.set_xlabel('Training-set size (compounds)')
ax.set_ylabel('Scaffold-split test ROC-AUC')
ax.legend(frameon=False, fontsize=8)
panel_label(ax, 'A')
fig.tight_layout()
save_fig(fig, 'figure_learning_curves', fig_dir)
_register('figure_learning_curves.png', fig_dir / 'figure_learning_curves.png')
_register('figure_learning_curves.pdf', fig_dir / 'figure_learning_curves.pdf')

# ---- Figure: scaffold-level performance breakdown (singleton vs
# non-singleton), full pool, combined representation -- Q2 core figure. ----
sb_full = scaffold_breakdown_df[
    (scaffold_breakdown_df['activity_pool'] == 'full') & (scaffold_breakdown_df['feature_representation'] == 'combined') &
    (scaffold_breakdown_df['scaffold_stratum'] != 'overall') & scaffold_breakdown_df['classification_roc_auc'].notna()
]
fig, ax = plt.subplots(figsize=(6, 5))
x = np.arange(len(TARGETS))
width = 0.35
for i, stratum in enumerate(['singleton_scaffold', 'non_singleton_scaffold']):
    sub = sb_full[sb_full['scaffold_stratum'] == stratum].set_index('target').reindex(TARGETS)
    ax.bar(x + (i - 0.5) * width, sub['classification_roc_auc'], width,
           label=stratum.replace('_', ' '), color=['#0072B2', '#D55E00'][i], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([t.upper() for t in TARGETS])
ax.set_ylabel('Classification ROC-AUC')
ax.legend(frameon=False)
panel_label(ax, 'A')
fig.tight_layout()
save_fig(fig, 'figure_scaffold_performance_breakdown', fig_dir)
_register('figure_scaffold_performance_breakdown.png', fig_dir / 'figure_scaffold_performance_breakdown.png')
_register('figure_scaffold_performance_breakdown.pdf', fig_dir / 'figure_scaffold_performance_breakdown.pdf')
print('Figures saved: figure_learning_curves, figure_scaffold_performance_breakdown '
      '(plot-source data already in learning_curves.csv / scaffold_performance_breakdown.csv)')





## Final Manifest

Single source-of-truth record tying together every output this notebook
produced: config, inputs (notebook 02's frozen files), outputs (every CSV/
JSON/figure registered via `_register()` across all 8 modules), SHA-256
hashes, package versions, timestamp, and best-effort git commit.



In [ ]:
FULL_CONFIG = {
    'targets': TARGETS,
    'pools': POOLS,
    'run_combinations': [f'{target}_{pool}' for target, pool in RUN_COMBOS],
    'gpcr_combos_env': os.environ.get('GPCR_COMBOS', ''),
    'feature_representations': FEATURE_REPS,
    'algorithms': ALGORITHMS,
    'morgan_radius': MORGAN_RADIUS,
    'morgan_n_bits': MORGAN_NBITS,
    'descriptor_columns': DESCRIPTOR_COLS,
    'activity_threshold': ACTIVITY_THRESHOLD,
    'tuning_mode': TUNING_MODE,
    'fixed_hyperparameters': FIXED_HYPERPARAMS if TUNING_MODE == 'fixed' else None,
    'n_optuna_trials_per_target_pool_model_task': N_OPTUNA_TRIALS if TUNING_MODE == 'optuna' else 0,
    'optuna_cv_folds': OPTUNA_CV_FOLDS,
    'optuna_parallel_trials': OPTUNA_PARALLEL_TRIALS if TUNING_MODE == 'optuna' else 0,
    'optuna_cv_n_jobs': OPTUNA_CV_N_JOBS if TUNING_MODE == 'optuna' else 0,
    'hyperparameter_tuning_scope': 'tuned once per (target, pool) on the combined representation, reused across all 3 representations',
    'n_repeated_scaffold_splits': N_REPEATS,
    'test_size': TEST_SIZE,
    'y_randomization_iterations': Y_RAND_ITERATIONS,
    'random_seed': RANDOM_STATE,
}

manifest_inputs = {
    'manifest_02_data_cleaning.json': {'path': str(manifest_02_path)},
    'compound_scaffold_assignments.csv': {'path': str(processed_path / 'compound_scaffold_assignments.csv')},
    'dataset_summary_all_targets_pools.csv': {'path': str(processed_path / 'dataset_summary_all_targets_pools.csv')},
}
for target, pool in RUN_COMBOS:
        fname = f'cleaned_data_{target}_{pool}.csv'
        manifest_inputs[fname] = {'path': str(processed_path / fname)}

write_manifest(
    results_path / shard_name('manifest_03_ml_benchmark.json'),
    config_summary=FULL_CONFIG,
    outputs=all_outputs,
    inputs=manifest_inputs,
)







## Summary

**Bugs fixed , cross-checked against
the prior audit:**

1. **Target/pool asymmetry removed entirely.** No `is_cb2`-style branching
   anywhere — every (target, pool) combination runs the same nested-CV
   protocol (same outer-fold count, same 50 Optuna trials tuned inside each
   fold, same 3 feature representations), with no per-target asymmetry.
2. **Hyperparameter policy is explicit and leak-free.** Default `TUNING_MODE
   = 'optuna'` tunes leak-free inside each fold; an opt-in `TUNING_MODE =
   'fixed'` uses documented FIXED_HYPERPARAMS instead (a no-tuning run, still
   leak-free since nothing is selected). Unlike an earlier implementation's
   `get_default_hyperparameters()`, BOTH paths carry class weighting (loaded from cache if already tuned, tuned fresh otherwise).
   class_weight/scale_pos_weight are tuned AND are on the only path that
   produces a reported model (lesson 2 fixed structurally, not just patched).
3. **Y-randomization takes the actual best-performing model as a parameter**
   (`run_y_randomization(model_builder_fn, algo_key, ...)`), determined from
   Module D's repeated-benchmark results, never hardcoded to XGBoost
   (lesson 1).
4. **Applicability domain computed once per (target, pool, representation)**
   and frozen as a single joblib artifact
   (`ml/models/{target}_{pool}/{representation}/ad_reference.pkl`), loaded
   read-only by every downstream stage — no redundant refit (lesson 15).
5. **Package versions enumerated programmatically** from the actual import
   list (`_capture_package_versions()`), not a fixed hardcoded list (lesson 0b).
6. **Every bootstrap/SHAP/plot-source array is persisted**, not just the
   rendered figure — SHAP value matrices and explained-sample arrays are
   saved as `.npy` (an earlier implementation removed these before
   saving, deleting the data that produced its own SHAP plots).
7. **Full model names everywhere** (`Random Forest`/`XGBoost`/`LightGBM`),
   never `RF`/`XGB`/`LGB` abbreviations, in every label, filename, and
   printed string.
8. **No on-figure titles anywhere** — `panel_label()` bold letters only, per
   notebook 02's convention; captions are the manuscript's job.
9. **No numbered figure/table references** in any printed or saved text
   (an earlier implementation printed an outdated table reference
   twice — no equivalent here).
10. **`PROJECT_DIR` correctly points at `gpcr_benchmark`**, not the
    an obsolete target-specific path.
11. **Scaffold groups reused from notebook 02's `global_scaffold_id`**,
    never recomputed via RDKit independently — avoids silent drift between
    this notebook's train/test split and notebook 02's diversity reporting.
12. **Effect sizes reported alongside every Wilcoxon p-value**
    (rank-biserial correlation) — lesson 24.
13. **Cross-validation and calibration/conformal fitting are wired
    explicitly to the training partition** (`X[train_idx]`), with a
    separate calibration holdout carved OUT OF training only — the test set
    is never touched for model selection, calibration, or conformal
    fitting (lessons 13/14).

**Judgment calls made in this build (flag for review):**
- Hyperparameters are tuned once per (target, pool) on the **combined**
  representation and reused across all 3 feature representations for the
  ablation comparison, rather than re-tuning per representation — this
  isolates the feature-representation effect from a re-discovered-
  hyperparameter effect, and mirrors an earlier implementation's own
  tune-once-reuse-across-ablation-variants design (only the CB2/CB1 trial-
  count asymmetry was removed, not this structural choice).
- `OPTUNA_CV_FOLDS = 5` inside each Optuna trial (an earlier implementation's config
  listed `cv_folds: 20` for tuning, but that number was never actually
  exercised since `run_hyperparameter_tuning` defaulted to `False` there) —
  5-fold scaffold `GroupKFold` per trial is a reasonable, defensible choice
  at a 50-trial budget; revisit if compute allows more.
- Applicability domain is computed per (target, pool, **representation**)
  — Module E's brief gave an illustrative path without the representation
  subfolder, but the module's own text requires per-representation AD, so
  the path was extended to `ml/models/{target}_{pool}/{representation}/ad_reference.pkl`.
- Error analysis (persistent misclassification, difficulty analysis)
  re-runs the repeated scaffold splits once more for only the best
  classifier (not all 3 algorithms), since per-compound outcome tracking
  is the one place that level of detail is actually used — Module D's
  main repeated-benchmark loop tracks aggregate metrics and native feature
  importances only, to avoid bloating every repeat with data most
  combinations never consume.
- This notebook does not reproduce an earlier implementation's Colab-disconnect
  `run_single_pool()`/`main()` resumability wrapper wholesale — Module D's
  expensive Optuna tuning step is checkpointed (skips already-tuned
  combinations), but Modules A/B/C/E/F/G/H run straight through. Given HPC
  availability), this trades some resumability for
  simpler, more auditable code; revisit if long-running HPC jobs need
  checkpoint/resume.
